# Hamiltonian Yamada Phase Maps

> **Role:** advanced, compute-intensive application and publication-reproduction
> notebook. It is not a beginner tutorial and is not executed in the ordinary
> pull-request notebook matrix.

Use this notebook when your input is a family of in-memory nodal Hamiltonians or
Bloch-vector fields and you want to turn a two-parameter scan into an auditable
Yamada-topology phase map. Install the `nodal`, `viz`, and `notebook` extras and
use a compute environment for full regeneration.

The production route is

\[
(\lambda,\Gamma) \longrightarrow H(\mathbf{k};\lambda,\Gamma)
\longrightarrow \text{filled exceptional region}
\longrightarrow \text{skeleton} \longrightarrow G\subset\mathbb R^3
\longrightarrow \Upsilon(G;Y).
\]

The configured study evaluates five transitions on up to 60 lambda samples and
50 candidate Gamma samples with a $120^3$ volume per evaluated cell. Row caches
reduce repeated work, but a clean full run remains a substantial workload. Do
not run all cells on a login node.

The notebook keeps audit steps visible: per-cell error records, connected-region
stabilization, classic and contraction-equivalent classifications, endpoint
checks, static figures, and an interactive Plotly view of representative
exceptional surfaces and their graph skeletons. Generated outputs live below
`User_guide/applications/results/06_hamiltonian_yamada_phase_maps`; execution
outputs are deliberately not stored in the notebook JSON.

For a guided explanation and a directly viewable interactive artifact, read the
website page **Hamiltonian Yamada Phase Maps** before regenerating this notebook.


In [ ]:
from __future__ import annotations

import csv
import dataclasses
import functools
import hashlib
import itertools
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

# Keep native numerical libraries from oversubscribing inside joblib workers.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import networkx as nx
import numpy as np
import sympy as sp
from joblib import Parallel, delayed
from IPython.display import IFrame, display
from skimage.measure import euler_number, label as label_volume

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "knotted_graph").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root above "
        f"the current working directory: {start}"
    )


ROOT = find_repo_root(Path.cwd().resolve())
import knotted_graph
from pathlib import Path
import sys

import knotted_graph
from knotted_graph.invariants.yamada.native import (
    native_available,
    native_import_error,
)

print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())

from knotted_graph.applications.nodal.models import (
    hopf_link_bloch_vector,
    pq_torus_knot_bloch_vector,
    solomon_bloch_vector,
    threelink_bloch_vector,
    trefoil_bloch_vector,
    unknot_bloch_vector,
)
from knotted_graph.applications.nodal.deformation import NodalBlochPath
from knotted_graph.applications.nodal.skeleton import NodalSkeleton
from knotted_graph.inputs import KnotFunction
from knotted_graph.applications.phase_maps import make_yamada_phase_map
from knotted_graph.core import idx_to_coord, remove_leaf_nodes, simplify_edges, smooth_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.invariants.yamada import compute_graph_yamada_polynomial
from knotted_graph.invariants.yamada.compact import CompactGraph
from knotted_graph.projection import compute_yamada_polynomial

kg_file = Path(knotted_graph.__file__).resolve()

A = sp.Symbol("A")
CACHE_VERSION = "hamiltonian_yamada_phase_maps_nodal_only_v4_application_results"
RESULT_DIR = ROOT / "User_guide" / "applications" / "results" / "06_hamiltonian_yamada_phase_maps"
FIGURE_DIR = RESULT_DIR / "figures"
PHASE_REPORT_CSV = RESULT_DIR / "06_hamiltonian_yamada_phase_map_report.csv"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Default grid: dense enough for phase-map clarity, but still practical with caching.
PHASE_LAMBDAS = np.linspace(0.0, 1.0, 60)
PHASE_GAMMAS = np.linspace(0.30, 5.25, 50)
ENDPOINT_DISTINCTION_GAMMA_MAX = 1.35
SKELETON_DIMENSION = 120
N_JOBS = 2
SMOOTHING_RETRIES = (4.0, 1.0, 0.25, 0.0)
PROJECTION_RETRY_SAMPLES = (32, 96, 256)

# Raw Yamada is still saved, but paper plots use only spatially supported
# components. Tiny islands are a numerical skeletonization diagnostic, not a
# believable standalone topological phase. A region must cover at least 1% of
# the displayed parameter grid, with a 5-cell lower bound.
STABLE_MIN_COMPONENT_FRACTION = 0.01


def stable_min_component_cells(lambdas=PHASE_LAMBDAS, gammas=PHASE_GAMMAS) -> int:
    return max(
        5,
        int(np.ceil(STABLE_MIN_COMPONENT_FRACTION * len(lambdas) * len(gammas))),
    )


STABLE_MIN_COMPONENT_CELLS = stable_min_component_cells()

print("knotted_graph source:", kg_file)
print("candidate grid:", len(PHASE_LAMBDAS), "lambda samples x", len(PHASE_GAMMAS), "Gamma samples")
print("dimension:", SKELETON_DIMENSION, "n_jobs:", N_JOBS)
print("endpoint distinction reference Gamma max:", ENDPOINT_DISTINCTION_GAMMA_MAX)
print("results:", RESULT_DIR)
print("figures:", FIGURE_DIR)


## Configuration, cost, and provenance

The setup cell fixes numerical-thread limits, resolves the repository, reports
the imported package and native Yamada backend, defines output/cache locations,
and declares the finite sampling grid. Inspect its printed values before any
scan. In particular:

- `PHASE_LAMBDAS` and `PHASE_GAMMAS` define the finite grid; changing them changes
  the phase-map evidence.
- `SKELETON_DIMENSION` controls the three-dimensional voxel resolution and is a
  dominant time/memory parameter.
- `N_JOBS` controls row-level concurrency, while each Yamada calculation uses an
  explicit single-worker setting to avoid nested oversubscription.
- cached rows are accepted only through a digest of the transition, axes,
  resolution, and retry policy.

A colored cell is therefore a recorded finite-resolution calculation, not an
analytic continuum phase boundary.


In [ ]:
def configure_reference_style() -> None:
    mpl.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.labelsize": 20,
            "axes.labelweight": "bold",
            "axes.titlesize": 14,
            "xtick.labelsize": 15,
            "ytick.labelsize": 15,
            "figure.dpi": 140,
            "savefig.dpi": 350,
        }
    )


def torus_2_5_bloch_vector(gamma: float):
    return pq_torus_knot_bloch_vector(2, 5, gamma)


TRANSITIONS = [
    {
        "key": "hopf_to_trefoil",
        "title": "Hopf link to trefoil",
        "start_name": "Hopf link",
        "end_name": "Trefoil",
        "start": hopf_link_bloch_vector,
        "end": trefoil_bloch_vector,
    },
    {
        "key": "hopf_to_solomon",
        "title": "Hopf link to Solomon link",
        "start_name": "Hopf link",
        "end_name": "Solomon link",
        "start": hopf_link_bloch_vector,
        "end": solomon_bloch_vector,
    },
    {
        "key": "unknot_to_trefoil",
        "title": "Unknot to trefoil",
        "start_name": "Unknot",
        "end_name": "Trefoil",
        "start": unknot_bloch_vector,
        "end": trefoil_bloch_vector,
    },
    {
        "key": "unknot_to_solomon",
        "title": "Unknot to Solomon link",
        "start_name": "Unknot",
        "end_name": "Solomon link",
        "start": unknot_bloch_vector,
        "end": solomon_bloch_vector,
    },
    {
        "key": "trefoil_to_cinquefoil",
        "title": "Trefoil to cinquefoil",
        "start_name": "Trefoil",
        "end_name": "Cinquefoil",
        "start": trefoil_bloch_vector,
        "end": torus_2_5_bloch_vector,
    },
]

TRANSITION_PATHS = {
    spec["key"]: NodalBlochPath(
        spec["start"], spec["end"],
        start_name=spec["start_name"], end_name=spec["end_name"],
    )
    for spec in TRANSITIONS
}

def hamiltonian_matrix_from_bloch_vector(vector):
    dx, dy, dz = vector
    return sp.ImmutableDenseMatrix([[dz, dx - sp.I * dy], [dx + sp.I * dy, -dz]])


example_path = NodalBlochPath(hopf_link_bloch_vector, trefoil_bloch_vector)
example_vector = example_path.at(gamma=0.50, lam=0.25)
print("Example H(lambda=0.25, Gamma=0.50):")
display(hamiltonian_matrix_from_bloch_vector(example_vector))


## Reusable phase-map API

The production scan below keeps the paper-specific audits and connected-region
stabilization. The reusable library entry point is `make_yamada_phase_map`.

This notebook demonstrates:

- `source_kind="nodal"`: two \(2\times2\) Hamiltonians, Bloch vectors, a
  `NodalBlochPath`, or endpoint factories depending on the second-axis parameter.
- `source_kind="knot"`: two `KnotFunction` objects or a `KnotFunctionPath`; the
  second axis is the tube/level radius.

For nodal scans, `force_genus_zero_vertex=True` is the default: a closed genus-zero
filled region is classified as the one-vertex Yamada phase before graph extraction.


In [ ]:
# Reusable wrappers for knot-field and nodal-Hamiltonian phase maps.

def yamada_phase_map_from_knot_functions(
    start: KnotFunction,
    end: KnotFunction,
    *,
    lambdas=np.linspace(0.0, 1.0, 21),
    radii=np.linspace(0.05, 0.35, 16),
    dimension=96,
    span=((-4.0, 4.0),) * 3,
):
    return make_yamada_phase_map(
        start,
        end,
        source_kind="knot",
        lambdas=lambdas,
        parameters=radii,
        parameter_name="radius",
        dimension=dimension,
        span=span,
        yamada_options={"n_jobs": 1, "num_rotation_samples": 32, "method": "negami"},
    )


def yamada_phase_map_from_nodal_hamiltonians(
    H0,
    H1,
    *,
    lambdas=np.linspace(0.0, 1.0, 21),
    gammas=np.linspace(0.30, 2.50, 21),
    dimension=96,
    span=None,
):
    return make_yamada_phase_map(
        H0,
        H1,
        source_kind="nodal",
        lambdas=lambdas,
        parameters=gammas,
        parameter_name=r"$\Gamma$",
        dimension=dimension,
        span=span,
        yamada_options={"n_jobs": 1, "num_rotation_samples": 32, "method": "negami"},
    )


RUN_UNIVERSAL_PHASE_MAP_EXAMPLES = False

if RUN_UNIVERSAL_PHASE_MAP_EXAMPLES:
    trefoil_field = KnotFunction.from_name("3_1")
    figure8_field = KnotFunction.from_name("4_1")

    knot_result = yamada_phase_map_from_knot_functions(
        trefoil_field,
        figure8_field,
        lambdas=np.linspace(0, 1, 5),
        radii=np.linspace(0.08, 0.20, 4),
        dimension=48,
    )

    nodal_result = yamada_phase_map_from_nodal_hamiltonians(
        hopf_link_bloch_vector,
        trefoil_bloch_vector,
        lambdas=np.linspace(0, 1, 5),
        gammas=np.linspace(0.30, 0.90, 4),
        dimension=48,
    )

    display(knot_result.phase_grid()[0])
    display(nodal_result.phase_grid()[0])


## What one phase-map record means

Each successful cell stores the sampled parameters, graph size, connected
components, cycle rank, degree sequence, total edge-geometry samples, exact
Yamada expression, and a stable phase signature. A failed extraction stores an
error instead of silently becoming a zero invariant.

`make_yamada_phase_map(...)` is the reusable application API. The longer code
below adds the publication-specific extraction retries, cache layout,
stabilization, figures, and validation rules. Keep that distinction in mind when
copying code into a new project.


In [ ]:
@dataclasses.dataclass(frozen=True)
class HamiltonianPhaseRecord:
    transition: str
    title: str
    lam: float
    gamma: float
    source: str
    core_mode: str
    nodes: int
    edges: int
    components: int
    cycle_rank: int
    degree_sequence: tuple[int, ...]
    phase_signature: str
    polynomial: str
    error: str | None = None


def canonical_polynomial(polynomial: sp.Expr) -> sp.Expr:
    try:
        return sp.factor(sp.together(sp.expand(polynomial)))
    except Exception:
        return sp.expand(polynomial)


def polynomial_signature(polynomial: sp.Expr) -> str:
    return sp.srepr(canonical_polynomial(polynomial))


def graph_summary(graph: nx.MultiGraph) -> tuple[int, int, int, int, tuple[int, ...]]:
    nodes = graph.number_of_nodes()
    edges = graph.number_of_edges()
    components = nx.number_connected_components(graph) if nodes else 0
    cycle_rank = edges - nodes + components
    degrees = tuple(sorted((degree for _, degree in graph.degree()), reverse=True))
    return nodes, edges, components, cycle_rank, degrees


def one_vertex_graph() -> nx.MultiGraph:
    graph = nx.MultiGraph()
    graph.add_node(0)
    return graph


def interior_boundary_faces(mask: np.ndarray) -> list[str]:
    mask = np.asarray(mask, dtype=bool)
    faces = []
    for label, touched in [
        ("kx_min", mask[0, :, :].any()),
        ("kx_max", mask[-1, :, :].any()),
        ("ky_min", mask[:, 0, :].any()),
        ("ky_max", mask[:, -1, :].any()),
        ("kz_min", mask[:, :, 0].any()),
        ("kz_max", mask[:, :, -1].any()),
    ]:
        if bool(touched):
            faces.append(label)
    return faces


def interior_topology_summary(skeleton: NodalSkeleton) -> dict:
    mask = np.asarray(skeleton._interior_mask, dtype=bool)
    if not mask.any():
        return {
            "interior_voxels": 0,
            "components": 0,
            "euler_characteristic": 0,
            "handle_rank": 0,
            "boundary_faces": [],
            "touches_boundary": False,
            "closed_in_window": False,
            "forces_vertex": True,
        }
    _, component_count = label_volume(mask, connectivity=3, return_num=True)
    euler = int(euler_number(mask, connectivity=3))
    boundary_faces = interior_boundary_faces(mask)
    handle_rank = max(0, int(component_count) - euler)
    closed_in_window = len(boundary_faces) == 0
    return {
        "interior_voxels": int(mask.sum()),
        "components": int(component_count),
        "euler_characteristic": euler,
        "handle_rank": int(handle_rank),
        "boundary_faces": boundary_faces,
        "touches_boundary": bool(boundary_faces),
        "closed_in_window": bool(closed_in_window),
        "forces_vertex": bool(closed_in_window and handle_rank == 0),
    }


def topology_forces_vertex(skeleton: NodalSkeleton) -> bool:
    return bool(interior_topology_summary(skeleton)["forces_vertex"])


def skeleton_failure_forces_vertex(exc: Exception) -> bool:
    message = str(exc)
    return any(
        fragment in message
        for fragment in (
            "graph has no edges",
            "skeleton image is empty",
            "does not contain any True voxels",
            "collapsed to fewer than two distinct points",
            "Skeletonization produced no points",
        )
    )


def _drop_consecutive_duplicate_points(points, *, atol: float = 1e-10) -> np.ndarray:
    pts = np.asarray(points, dtype=float)
    if len(pts) == 0:
        return pts
    keep = [0]
    for index in range(1, len(pts)):
        if not np.allclose(pts[index], pts[keep[-1]], atol=atol, rtol=0.0):
            keep.append(index)
    return pts[np.asarray(keep, dtype=int)]


def drop_degenerate_edges(graph: nx.MultiGraph, *, atol: float = 1e-10) -> nx.MultiGraph:
    H = graph.copy()
    for u, v, key, data in list(H.edges(keys=True, data=True)):
        pts = data.get("pts")
        if pts is None:
            start = np.asarray(H.nodes[u].get("pos"), dtype=float)
            end = np.asarray(H.nodes[v].get("pos"), dtype=float)
            is_degenerate = (
                start.shape != (3,)
                or end.shape != (3,)
                or not np.isfinite(start).all()
                or not np.isfinite(end).all()
                or np.allclose(start, end, atol=atol, rtol=0.0)
            )
        else:
            pts_arr = np.asarray(pts, dtype=float)
            is_degenerate = (
                pts_arr.ndim != 2
                or pts_arr.shape[1] != 3
                or pts_arr.shape[0] < 2
                or not np.isfinite(pts_arr).all()
                or len(_drop_consecutive_duplicate_points(pts_arr, atol=atol)) < 2
            )
        if is_degenerate:
            H.remove_edge(u, v, key)
    if H.number_of_edges() > 0:
        H.remove_nodes_from([node for node, degree in H.degree() if degree == 0])
    return H


def core_candidates_from_skeleton(skeleton: NodalSkeleton):
    try:
        raw_graph = skeleton_image_to_graph(skeleton._skeleton_image)
    except Exception as exc:
        if skeleton_failure_forces_vertex(exc):
            return [("vertex-empty-skeleton", one_vertex_graph())]
        raise
    raw_graph = drop_degenerate_edges(raw_graph)
    if raw_graph.number_of_edges() == 0:
        return [("vertex-degenerate-skeleton", one_vertex_graph())]
    core = remove_leaf_nodes(raw_graph)
    core = drop_degenerate_edges(core)
    if core.number_of_edges() == 0:
        return [("vertex-after-prune", one_vertex_graph())]

    try:
        core = simplify_edges(core)
    except Exception as exc:
        if not skeleton_failure_forces_vertex(exc):
            raise
        core = drop_degenerate_edges(core)
        if core.number_of_edges() == 0:
            return [("vertex-after-degenerate-simplify", one_vertex_graph())]
    core = drop_degenerate_edges(core)
    if core.number_of_edges() == 0:
        return [("vertex-after-simplify", one_vertex_graph())]

    topology = interior_topology_summary(skeleton)
    if topology["forces_vertex"]:
        return [("vertex-genus-zero-surface", one_vertex_graph())]

    candidates = []
    for epsilon in SMOOTHING_RETRIES:
        try:
            smoothed = smooth_edges(core, epsilon=epsilon, copy=True)
        except Exception:
            continue
        candidates.append((f"smooth-{epsilon:g}", smoothed))
    if not candidates:
        candidates.append(("unsmoothed", core))
    return candidates


def is_closed_link_or_knot_core(graph: nx.MultiGraph) -> bool:
    nodes, edges, components, cycle_rank, degrees = graph_summary(graph)
    return edges > 0 and cycle_rank == components and all(degree == 2 for degree in degrees)


def should_use_embedded_yamada(graph: nx.MultiGraph) -> bool:
    return is_closed_link_or_knot_core(graph) or graph.number_of_edges() <= 3


def evaluate_yamada_on_candidates(candidates):
    first_mode, first_graph = candidates[0]
    if first_graph.number_of_edges() == 0:
        polynomial = compute_graph_yamada_polynomial(first_graph, A)
        return "vertex", first_mode, first_graph, polynomial

    if any(should_use_embedded_yamada(graph) for _, graph in candidates):
        last_error = None
        for mode, graph in candidates:
            if any(degree == 1 for _, degree in graph.degree()):
                continue
            if not should_use_embedded_yamada(graph):
                continue
            for sample_count in PROJECTION_RETRY_SAMPLES:
                try:
                    result = compute_yamada_polynomial(
                        graph,
                        A,
                        normalize=True,
                        n_jobs=1,
                        num_rotation_samples=sample_count,
                        crossing_warning_threshold=999,
                        method="negami",
                        return_result=True,
                    )
                    return "embedded", mode, graph, result.polynomial
                except Exception as exc:
                    last_error = exc

        # This is still an exact Yamada value from the target library. It is kept
        # visible in the `source` column so projection fallback is auditable.
        polynomial = compute_graph_yamada_polynomial(first_graph, A)
        return "graph-fallback", first_mode, first_graph, polynomial

    polynomial = compute_graph_yamada_polynomial(first_graph, A)
    return "graph", first_mode, first_graph, polynomial


def evaluate_phase_cell(spec, gamma: float, lam: float) -> HamiltonianPhaseRecord:
    transition_path = TRANSITION_PATHS[spec["key"]]
    try:
        bloch_vector = transition_path.at(float(gamma), float(lam))
        skeleton = NodalSkeleton(char=bloch_vector, dimension=SKELETON_DIMENSION)
        candidates = core_candidates_from_skeleton(skeleton)
        source, mode, graph, polynomial = evaluate_yamada_on_candidates(candidates)
        nodes, edges, components, cycle_rank, degrees = graph_summary(graph)
        if any(degree == 1 for degree in degrees):
            raise ValueError(f"leaf degree remained after pruning: {degrees}")
        poly = canonical_polynomial(polynomial)
        return HamiltonianPhaseRecord(
            transition=spec["key"], title=spec["title"],
            lam=float(lam), gamma=float(gamma), source=source, core_mode=mode,
            nodes=nodes, edges=edges, components=components,
            cycle_rank=cycle_rank, degree_sequence=degrees,
            phase_signature="yamada:" + sp.srepr(poly), polynomial=sp.sstr(poly),
            error=None,
        )
    except Exception as exc:
        return HamiltonianPhaseRecord(
            transition=spec["key"], title=spec["title"],
            lam=float(lam), gamma=float(gamma), source="error", core_mode="error",
            nodes=0, edges=0, components=0, cycle_rank=0, degree_sequence=(),
            phase_signature=f"error:{type(exc).__name__}:{exc}", polynomial="",
            error=f"{type(exc).__name__}: {exc}",
        )


In [ ]:
def cache_digest(spec, lambdas, gammas) -> str:
    payload = {
        "version": CACHE_VERSION,
        "transition": spec["key"],
        "start": spec["start_name"],
        "end": spec["end_name"],
        "lambdas": [round(float(value), 10) for value in lambdas],
        "gammas": [round(float(value), 10) for value in gammas],
        "dimension": int(SKELETON_DIMENSION),
        "smoothing_retries": list(SMOOTHING_RETRIES),
        "projection_retry_samples": list(PROJECTION_RETRY_SAMPLES),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


def record_to_json(record: HamiltonianPhaseRecord) -> dict:
    row = dataclasses.asdict(record)
    row["degree_sequence"] = list(record.degree_sequence)
    return row


def record_from_json(row: dict) -> HamiltonianPhaseRecord:
    row = dict(row)
    row["degree_sequence"] = tuple(int(value) for value in row.get("degree_sequence", []))
    return HamiltonianPhaseRecord(**row)


def scan_gamma_row(spec, gamma: float, lambdas) -> list[HamiltonianPhaseRecord]:
    return [evaluate_phase_cell(spec, float(gamma), float(lam)) for lam in lambdas]


def compute_gamma_row_for_cache(spec, row_index: int, gamma: float, lambdas, digest: str):
    gamma_value = float(gamma)
    row_t0 = time.perf_counter()
    row_records = scan_gamma_row(spec, gamma_value, lambdas)
    row_elapsed = time.perf_counter() - row_t0
    return int(row_index), gamma_value, row_elapsed, row_records


def write_gamma_row_cache(path: Path, spec, digest: str, row_index: int, gamma_value: float, row_elapsed: float, row_records) -> None:
    row_payload = {
        "version": CACHE_VERSION,
        "digest": digest,
        "transition": spec["key"],
        "gamma_index": int(row_index),
        "gamma": float(gamma_value),
        "dimension": int(SKELETON_DIMENSION),
        "elapsed_seconds": float(row_elapsed),
        "records": [record_to_json(record) for record in row_records],
    }
    tmp_path = path.with_suffix(".tmp")
    tmp_path.write_text(json.dumps(row_payload, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def gammas_from_records(records) -> np.ndarray:
    return np.asarray(sorted({round(float(record.gamma), 10) for record in records}), dtype=float)


def record_sample_key(records, lambdas=PHASE_LAMBDAS, gammas=None) -> tuple:
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    return (
        len(records),
        tuple(round(float(value), 10) for value in lambdas),
        tuple(round(float(value), 10) for value in gammas),
    )


def is_one_vertex_yamada_record(record: HamiltonianPhaseRecord) -> bool:
    return (
        record.source == "vertex"
        or record.polynomial == "-1"
        or (int(record.nodes) == 1 and int(record.edges) == 0 and int(record.cycle_rank) == 0)
    )


def gamma_row_is_all_vertex(row_records) -> bool:
    return bool(row_records) and all(is_one_vertex_yamada_record(record) for record in row_records)


def terminal_transition_gammas_from_records(records, lambdas=PHASE_LAMBDAS, gammas=PHASE_GAMMAS) -> tuple[np.ndarray, int]:
    lookup = {(round(float(record.gamma), 10), round(float(record.lam), 10)): record for record in records}
    for row_index, gamma in enumerate(gammas):
        gamma_key = round(float(gamma), 10)
        row_records = []
        for lam in lambdas:
            record = lookup.get((gamma_key, round(float(lam), 10)))
            if record is None:
                if row_index == 0:
                    raise RuntimeError("records are missing the low-Gamma baseline row")
                raise RuntimeError(
                    "records ended before an all-lambda one-vertex Yamada row was found"
                )
            row_records.append(record)
        if gamma_row_is_all_vertex(row_records):
            return np.asarray([float(value) for value in gammas[: row_index + 1]], dtype=float), row_index
    raise RuntimeError("candidate Gamma grid never reached an all-lambda one-vertex Yamada row")


def crop_records_to_gammas(records, gammas) -> list[HamiltonianPhaseRecord]:
    allowed = {round(float(value), 10) for value in gammas}
    return [record for record in records if round(float(record.gamma), 10) in allowed]


def cached_transition_scan(spec, lambdas=PHASE_LAMBDAS, gammas=PHASE_GAMMAS):
    digest = cache_digest(spec, lambdas, gammas)
    cache_path = RESULT_DIR / f"{spec['key']}_{digest}.json"
    if cache_path.exists():
        payload = json.loads(cache_path.read_text())
        records = [record_from_json(row) for row in payload["records"]]
        display_gammas, terminal_index = terminal_transition_gammas_from_records(records, lambdas, gammas)
        records = crop_records_to_gammas(records, display_gammas)
        print(
            f"loaded {spec['key']}: {cache_path.name}; "
            f"display Gamma max={display_gammas[-1]:.4f} "
            f"at first all-lambda vertex row {terminal_index + 1}/{len(gammas)}"
        )
        return records, cache_path

    row_cache_dir = RESULT_DIR / "row_cache" / spec["key"] / digest
    row_cache_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    rows = [None] * len(gammas)
    loaded_rows = 0
    terminal_index = None

    for row_index, gamma in enumerate(gammas):
        gamma_value = float(gamma)
        row_cache_path = row_cache_dir / f"gamma_{row_index:03d}.json"
        row_records = None
        if row_cache_path.exists():
            row_payload = json.loads(row_cache_path.read_text())
            row_valid = (
                row_payload.get("version") == CACHE_VERSION
                and row_payload.get("digest") == digest
                and int(row_payload.get("gamma_index", -1)) == row_index
                and abs(float(row_payload.get("gamma", np.nan)) - gamma_value) < 1e-10
                and len(row_payload.get("records", [])) == len(lambdas)
            )
            if row_valid:
                row_records = [record_from_json(record) for record in row_payload["records"]]
                loaded_rows += 1

        if row_records is None:
            _, gamma_value, row_elapsed, row_records = compute_gamma_row_for_cache(
                spec, row_index, gamma_value, lambdas, digest
            )
            write_gamma_row_cache(row_cache_path, spec, digest, row_index, gamma_value, row_elapsed, row_records)
            print(
                f"  computed {spec['key']} row {row_index + 1:02d}/{len(gammas)} "
                f"at Gamma={gamma_value:.4f}: {len(row_records)} cells in {row_elapsed:.2f}s"
            )

        rows[row_index] = row_records
        if gamma_row_is_all_vertex(row_records):
            terminal_index = row_index
            break

    if loaded_rows:
        print(f"  loaded {spec['key']}: {loaded_rows} cached Gamma rows before terminal crop")
    if terminal_index is None:
        raise RuntimeError(f"{spec['key']} never reached an all-lambda one-vertex Yamada row")

    display_rows = rows[: terminal_index + 1]
    records = [record for row in display_rows for record in row]
    elapsed = time.perf_counter() - t0
    payload = {
        "version": CACHE_VERSION,
        "transition": spec["key"],
        "title": spec["title"],
        "elapsed_seconds": elapsed,
        "candidate_gamma_count": len(gammas),
        "displayed_gamma_count": terminal_index + 1,
        "terminal_gamma_index": terminal_index,
        "terminal_gamma": float(gammas[terminal_index]),
        "terminal_rule": "first Gamma row for which every lambda cell has one-vertex Yamada",
        "records": [record_to_json(record) for record in records],
    }
    cache_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(
        f"computed {spec['key']}: {len(records)} displayed cells in {elapsed:.2f}s -> {cache_path.name}; "
        f"terminal Gamma={float(gammas[terminal_index]):.4f} ({terminal_index + 1}/{len(gammas)} rows)"
    )
    return records, cache_path


def records_to_grid(records, value_getter, lambdas=PHASE_LAMBDAS, gammas=None):
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    lookup = {(round(record.gamma, 10), round(record.lam, 10)): record for record in records}
    grid = np.empty((len(gammas), len(lambdas)), dtype=object)
    for i, gamma in enumerate(gammas):
        for j, lam in enumerate(lambdas):
            grid[i, j] = value_getter(lookup[(round(float(gamma), 10), round(float(lam), 10))])
    return grid


def phase_ids_for_records(records, lambdas=PHASE_LAMBDAS, gammas=None):
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    signatures = []
    seen = set()
    for record in records:
        if record.phase_signature not in seen:
            seen.add(record.phase_signature)
            signatures.append(record.phase_signature)
    signature_to_id = {signature: index + 1 for index, signature in enumerate(signatures)}
    labels = records_to_grid(
        records,
        lambda record: signature_to_id[record.phase_signature],
        lambdas=lambdas,
        gammas=gammas,
    ).astype(int)
    return labels, signature_to_id


def connected_label_components(labels: np.ndarray, *, connectivity: int = 4):
    labels = np.asarray(labels, dtype=int)
    if labels.ndim != 2:
        raise ValueError("labels must be a 2D phase grid")
    if connectivity == 8:
        offsets = [
            (-1, -1), (-1, 0), (-1, 1),
            (0, -1), (0, 1),
            (1, -1), (1, 0), (1, 1),
        ]
    elif connectivity == 4:
        offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    else:
        raise ValueError("connectivity must be 4 or 8")

    height, width = labels.shape
    visited = np.zeros(labels.shape, dtype=bool)
    component_index = -np.ones(labels.shape, dtype=int)
    components = []

    for row in range(height):
        for col in range(width):
            if visited[row, col]:
                continue
            label = int(labels[row, col])
            stack = [(row, col)]
            visited[row, col] = True
            cells = []
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in offsets:
                    rr, cc = r + dr, c + dc
                    if rr < 0 or rr >= height or cc < 0 or cc >= width:
                        continue
                    if visited[rr, cc] or int(labels[rr, cc]) != label:
                        continue
                    visited[rr, cc] = True
                    stack.append((rr, cc))
            index = len(components)
            components.append({"label": label, "cells": cells})
            for cell in cells:
                component_index[cell] = index
    return components, component_index


def label_component_stats(labels: np.ndarray, *, min_component_cells: int = STABLE_MIN_COMPONENT_CELLS) -> dict:
    components, _ = connected_label_components(labels, connectivity=4)
    sizes = [len(component["cells"]) for component in components]
    small_sizes = [size for size in sizes if size < min_component_cells]
    return {
        "classes": int(len(set(int(value) for value in np.ravel(labels)))),
        "components": int(len(components)),
        "small_components": int(len(small_sizes)),
        "small_cells": int(sum(small_sizes)),
        "max_component_cells": int(max(sizes) if sizes else 0),
    }


def stable_partition_labels(labels: np.ndarray, *, min_component_cells: int = STABLE_MIN_COMPONENT_CELLS):
    stable = np.asarray(labels, dtype=int).copy()
    raw_stats = label_component_stats(stable, min_component_cells=min_component_cells)
    reassigned_mask = np.zeros(stable.shape, dtype=bool)

    while True:
        components, component_index = connected_label_components(stable, connectivity=4)
        sizes = np.asarray([len(component["cells"]) for component in components], dtype=int)
        label_areas = Counter()
        for component, size in zip(components, sizes):
            label_areas[int(component["label"])] += int(size)

        changed = False
        for index, component in enumerate(components):
            if int(sizes[index]) >= min_component_cells:
                continue
            neighbor_votes = Counter()
            for r, c in component["cells"]:
                for dr in (-1, 0, 1):
                    for dc in (-1, 0, 1):
                        if dr == 0 and dc == 0:
                            continue
                        rr, cc = r + dr, c + dc
                        if rr < 0 or rr >= stable.shape[0] or cc < 0 or cc >= stable.shape[1]:
                            continue
                        neighbor_component = int(component_index[rr, cc])
                        if neighbor_component == index or sizes[neighbor_component] < min_component_cells:
                            continue
                        neighbor_votes[int(stable[rr, cc])] += 1
            if not neighbor_votes:
                continue

            replacement = max(
                neighbor_votes,
                key=lambda label: (neighbor_votes[label], label_areas[label], -label),
            )
            if replacement == int(component["label"]):
                continue
            for cell in component["cells"]:
                stable[cell] = replacement
                reassigned_mask[cell] = True
            changed = True

        if not changed:
            break

    stable_stats = label_component_stats(stable, min_component_cells=min_component_cells)
    audit = {
        "min_component_cells": int(min_component_cells),
        "raw_classes": raw_stats["classes"],
        "raw_components": raw_stats["components"],
        "raw_small_components": raw_stats["small_components"],
        "raw_small_cells": raw_stats["small_cells"],
        "stable_classes": stable_stats["classes"],
        "stable_components": stable_stats["components"],
        "stable_small_components": stable_stats["small_components"],
        "stable_small_cells": stable_stats["small_cells"],
        "reassigned_cells": int(reassigned_mask.sum()),
    }
    return stable, reassigned_mask, audit


def stable_phase_ids_for_records(records, lambdas=PHASE_LAMBDAS, gammas=None):
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    raw_labels, raw_signature_to_id = phase_ids_for_records(records, lambdas=lambdas, gammas=gammas)
    min_cells = stable_min_component_cells(lambdas, gammas)
    stable_raw_ids, reassigned_mask, audit = stable_partition_labels(raw_labels, min_component_cells=min_cells)
    id_to_signature = {phase_id: signature for signature, phase_id in raw_signature_to_id.items()}
    kept_raw_ids = sorted(set(int(value) for value in stable_raw_ids.ravel()))
    raw_id_to_stable_id = {raw_id: index + 1 for index, raw_id in enumerate(kept_raw_ids)}
    stable_labels = np.vectorize(raw_id_to_stable_id.__getitem__)(stable_raw_ids).astype(int)
    stable_signature_to_id = {
        id_to_signature[raw_id]: stable_id
        for raw_id, stable_id in raw_id_to_stable_id.items()
    }
    return stable_labels, stable_signature_to_id, stable_raw_ids, reassigned_mask, audit, raw_signature_to_id


def stable_region_ids(stable_labels: np.ndarray):
    components, _ = connected_label_components(stable_labels, connectivity=4)
    region_grid = np.zeros_like(stable_labels, dtype=int)
    for region_id, component in enumerate(components, start=1):
        for cell in component["cells"]:
            region_grid[cell] = region_id
    return region_grid, components


def short_polynomial(text: str, max_chars=120) -> str:
    return text if len(text) <= max_chars else text[: max_chars - 1] + "..."


PHASE_MODES = [
    {
        "key": "classic",
        "title": "Classic",
        "short_title": "Classic",
        "colorbar_title": "Yamada",
        "description": "Stable connected-region partition of exact Yamada signatures.",
    },
    {
        "key": "contraction",
        "title": "Up to contraction moves",
        "short_title": "Contraction",
        "colorbar_title": "Contraction",
        "description": "Classic regions are merged when either representative core graph can be reduced to the other by non-loop edge contractions.",
    },
]
PHASE_MODE_BY_KEY = {mode["key"]: mode for mode in PHASE_MODES}
_PHASE_MODE_CACHE: dict[tuple, dict] = {}


def one_vertex_compact_graph() -> CompactGraph:
    return CompactGraph(((0,),))


def compact_induced(compact: CompactGraph, nodes) -> CompactGraph:
    nodes = tuple(int(node) for node in nodes)
    if not nodes:
        return one_vertex_compact_graph()
    rows = tuple(tuple(compact.rows[i][j] for j in nodes) for i in nodes)
    return CompactGraph(rows)


def compact_drop_isolates(compact: CompactGraph) -> CompactGraph:
    if compact.n == 0:
        return one_vertex_compact_graph()
    keep = tuple(index for index in range(compact.n) if compact.degree(index) > 0)
    if not keep:
        return one_vertex_compact_graph()
    if len(keep) == compact.n:
        return compact
    return compact_induced(compact, keep)


def compact_prune_leaf_vertices(compact: CompactGraph) -> CompactGraph:
    compact = compact_drop_isolates(compact)
    while compact.edge_count:
        leaves = [index for index in range(compact.n) if compact.degree(index) == 1]
        if not leaves:
            return compact
        if len(leaves) == compact.n:
            return one_vertex_compact_graph()
        compact = compact_induced(compact, [index for index in range(compact.n) if index not in leaves])
        compact = compact_drop_isolates(compact)
    return one_vertex_compact_graph()


def compact_suppress_degree_two_vertices(compact: CompactGraph) -> CompactGraph:
    compact = compact_drop_isolates(compact)
    while compact.edge_count:
        candidates = [
            index
            for index in range(compact.n)
            if compact.degree(index) == 2 and compact.rows[index][index] == 0
        ]
        if not candidates:
            return compact
        compact = compact.suppress_degree_two(candidates[0])
        compact = compact_drop_isolates(compact)
    return one_vertex_compact_graph()


def compact_core_normal_form(compact: CompactGraph) -> CompactGraph:
    previous_rows = None
    compact = compact_drop_isolates(compact)
    while compact.rows != previous_rows:
        previous_rows = compact.rows
        compact = compact_prune_leaf_vertices(compact)
        compact = compact_suppress_degree_two_vertices(compact)
        compact = compact_drop_isolates(compact)
        if compact.edge_count == 0:
            return one_vertex_compact_graph()
    return compact


def compact_from_graph_core(graph: nx.MultiGraph) -> CompactGraph:
    topological = nx.MultiGraph()
    topological.add_nodes_from(graph.nodes())
    for u, v, _key in graph.edges(keys=True):
        topological.add_edge(u, v)
    if topological.number_of_nodes() == 0:
        topological.add_node(0)
    return compact_core_normal_form(CompactGraph.from_networkx(topological))


def compact_node_invariant(compact: CompactGraph, index: int) -> tuple:
    row = compact.rows[index]
    nonloop_multiplicities = sorted(row[j] for j in range(compact.n) if j != index and row[j])
    return (
        compact.degree(index),
        row[index],
        len(nonloop_multiplicities),
        tuple(nonloop_multiplicities),
    )


@functools.lru_cache(maxsize=None)
def compact_canonical_rows(rows: tuple[tuple[int, ...], ...]) -> tuple[tuple[int, ...], ...]:
    compact = compact_core_normal_form(CompactGraph(rows))
    n = compact.n
    if n == 0:
        return one_vertex_compact_graph().rows
    grouped: dict[tuple, list[int]] = {}
    for index in range(n):
        grouped.setdefault(compact_node_invariant(compact, index), []).append(index)
    groups = [tuple(grouped[key]) for key in sorted(grouped)]
    best = None
    for permuted_groups in itertools.product(*(itertools.permutations(group) for group in groups)):
        order = tuple(index for group in permuted_groups for index in group)
        candidate = tuple(tuple(compact.rows[order[i]][order[j]] for j in range(n)) for i in range(n))
        if best is None or candidate < best:
            best = candidate
    return best if best is not None else one_vertex_compact_graph().rows


def compact_canonical_signature(compact: CompactGraph) -> tuple[tuple[int, ...], ...]:
    return compact_canonical_rows(compact_core_normal_form(compact).rows)


@functools.lru_cache(maxsize=None)
def compact_contraction_closure_signatures_from_rows(rows: tuple[tuple[int, ...], ...]) -> frozenset:
    start_signature = compact_canonical_rows(rows)
    seen = {start_signature}
    stack = [CompactGraph(start_signature)]
    while stack:
        graph = stack.pop()
        for i in range(graph.n):
            for j in range(i + 1, graph.n):
                if graph.rows[i][j] <= 0:
                    continue
                child = compact_core_normal_form(graph.contract_edge(i, j))
                signature = compact_canonical_signature(child)
                if signature not in seen:
                    seen.add(signature)
                    stack.append(CompactGraph(signature))
    return frozenset(seen)


def compact_contraction_closure_signatures(compact: CompactGraph) -> frozenset:
    return compact_contraction_closure_signatures_from_rows(compact_canonical_signature(compact))


def compact_signature_text(signature: tuple[tuple[int, ...], ...]) -> str:
    return ";".join(",".join(str(value) for value in row) for row in signature)


def transition_path_for_spec(spec) -> NodalBlochPath:
    return NodalBlochPath(
        spec["start"], spec["end"],
        start_name=spec["start_name"], end_name=spec["end_name"],
    )


def skeleton_for_phase_record(spec, record: HamiltonianPhaseRecord):
    bloch_vector = transition_path_for_spec(spec).at(
        float(record.gamma),
        float(record.lam),
    )
    return NodalSkeleton(
        char=bloch_vector,
        dimension=SKELETON_DIMENSION,
    )


def representative_core_graph(skeleton: NodalSkeleton, record: HamiltonianPhaseRecord) -> nx.MultiGraph:
    try:
        candidates = core_candidates_from_skeleton(skeleton)
    except Exception as exc:
        if skeleton_failure_forces_vertex(exc):
            return one_vertex_graph()
        raise
    for mode, graph in candidates:
        if mode == record.core_mode:
            return graph
    return candidates[0][1]


def choose_representative_record(records, component, stable_raw_ids, reassigned_mask, raw_signature_to_id, lambdas=PHASE_LAMBDAS, gammas=None):
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    id_to_raw_signature = {phase_id: signature for signature, phase_id in raw_signature_to_id.items()}
    lookup = {(round(record.gamma, 10), round(record.lam, 10)): record for record in records}
    cells = list(component["cells"])
    coordinates = np.asarray(cells, dtype=float)
    centroid = coordinates.mean(axis=0)
    target_signature = id_to_raw_signature[int(stable_raw_ids[cells[0]])]

    choices = []
    for row_index, col_index in cells:
        record = lookup[(round(float(gammas[row_index]), 10), round(float(lambdas[col_index]), 10))]
        same_phase = record.phase_signature == target_signature
        preferred = same_phase and not bool(reassigned_mask[row_index, col_index]) and not record.error
        local_support = 0
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            rr, cc = row_index + dr, col_index + dc
            if 0 <= rr < stable_raw_ids.shape[0] and 0 <= cc < stable_raw_ids.shape[1]:
                local_support += int(stable_raw_ids[rr, cc] == stable_raw_ids[row_index, col_index])
        distance = float(np.linalg.norm(np.asarray([row_index, col_index], dtype=float) - centroid))
        choices.append((0 if preferred else 1, -local_support, distance, row_index, col_index, record))

    choices.sort(key=lambda item: item[:5])
    _, _, _, row_index, col_index, record = choices[0]
    return int(row_index), int(col_index), record


def build_classic_mode_data(records, spec, lambdas=PHASE_LAMBDAS, gammas=None) -> dict:
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    cache_key = (spec["key"], "classic", record_sample_key(records, lambdas=lambdas, gammas=gammas))
    if cache_key in _PHASE_MODE_CACHE:
        return _PHASE_MODE_CACHE[cache_key]

    stable_labels, stable_signature_to_id, stable_raw_ids, reassigned_mask, stability_audit, raw_signature_to_id = stable_phase_ids_for_records(
        records, lambdas=lambdas, gammas=gammas
    )
    region_grid, regions = stable_region_ids(stable_labels)
    id_to_raw_signature = {phase_id: signature for signature, phase_id in raw_signature_to_id.items()}
    region_info = {}
    for region_id, component in enumerate(regions, start=1):
        row_index, col_index, record = choose_representative_record(
            records, component, stable_raw_ids, reassigned_mask, raw_signature_to_id,
            lambdas=lambdas, gammas=gammas,
        )
        stable_raw_phase_id = int(stable_raw_ids[row_index, col_index])
        phase_id = int(stable_labels[row_index, col_index])
        region_info[region_id] = {
            "mode": "classic",
            "mode_title": PHASE_MODE_BY_KEY["classic"]["title"],
            "phase_id": phase_id,
            "region_id": int(region_id),
            "row_index": int(row_index),
            "col_index": int(col_index),
            "record": record,
            "stable_signature": id_to_raw_signature[stable_raw_phase_id],
            "classic_region_ids": [int(region_id)],
            "classic_phase_ids": [phase_id],
            "cell_count": int(len(component["cells"])),
        }

    data = {
        "mode": "classic",
        "mode_title": PHASE_MODE_BY_KEY["classic"]["title"],
        "labels": stable_labels,
        "region_grid": region_grid,
        "regions": regions,
        "region_info": region_info,
        "stable_signature_to_id": stable_signature_to_id,
        "stable_raw_ids": stable_raw_ids,
        "reassigned_mask": reassigned_mask,
        "raw_signature_to_id": raw_signature_to_id,
        "audit": {
            **stability_audit,
            "mode_classes": stability_audit["stable_classes"],
            "mode_components": stability_audit["stable_components"],
            "mode_small_components": stability_audit["stable_small_components"],
            "mode_small_cells": stability_audit["stable_small_cells"],
            "contraction_accessible_pairs": 0,
            "contraction_merged_classic_regions": 0,
        },
    }
    _PHASE_MODE_CACHE[cache_key] = data
    return data


class DisjointSet:
    def __init__(self, values):
        self.parent = {value: value for value in values}

    def find(self, value):
        parent = self.parent[value]
        if parent != value:
            parent = self.find(parent)
            self.parent[value] = parent
        return parent

    def union(self, left, right) -> bool:
        root_left = self.find(left)
        root_right = self.find(right)
        if root_left == root_right:
            return False
        if root_right < root_left:
            root_left, root_right = root_right, root_left
        self.parent[root_right] = root_left
        return True


def build_contraction_mode_data(records, spec, lambdas=PHASE_LAMBDAS, gammas=None) -> dict:
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    cache_key = (spec["key"], "contraction", record_sample_key(records, lambdas=lambdas, gammas=gammas))
    if cache_key in _PHASE_MODE_CACHE:
        return _PHASE_MODE_CACHE[cache_key]

    classic = build_classic_mode_data(records, spec, lambdas=lambdas, gammas=gammas)
    classic_region_ids = sorted(classic["region_info"])
    contraction_data = {}
    for region_id in classic_region_ids:
        info = classic["region_info"][region_id]
        record = info["record"]
        skeleton = skeleton_for_phase_record(spec, record)
        graph = representative_core_graph(skeleton, record)
        compact = compact_from_graph_core(graph)
        signature = compact_canonical_signature(compact)
        closure = compact_contraction_closure_signatures(compact)
        contraction_data[region_id] = {
            "signature": signature,
            "closure": closure,
            "closure_size": len(closure),
            "nodes": compact.n,
            "edges": compact.edge_count,
        }

    dsu = DisjointSet(classic_region_ids)
    accessible_pairs = []
    for left_index, left_id in enumerate(classic_region_ids):
        for right_id in classic_region_ids[left_index + 1 :]:
            left_signature = contraction_data[left_id]["signature"]
            right_signature = contraction_data[right_id]["signature"]
            left_to_right = right_signature in contraction_data[left_id]["closure"]
            right_to_left = left_signature in contraction_data[right_id]["closure"]
            if left_to_right or right_to_left:
                dsu.union(left_id, right_id)
                accessible_pairs.append({
                    "left_region_id": int(left_id),
                    "right_region_id": int(right_id),
                    "left_contracts_to_right": bool(left_to_right),
                    "right_contracts_to_left": bool(right_to_left),
                })

    groups: dict[int, list[int]] = {}
    for region_id in classic_region_ids:
        groups.setdefault(dsu.find(region_id), []).append(region_id)
    ordered_roots = sorted(groups, key=lambda root: min(groups[root]))
    root_to_phase_id = {root: index + 1 for index, root in enumerate(ordered_roots)}
    classic_region_to_phase_id = {
        region_id: root_to_phase_id[dsu.find(region_id)]
        for region_id in classic_region_ids
    }
    contraction_labels = np.vectorize(lambda region_id: classic_region_to_phase_id[int(region_id)])(
        classic["region_grid"]
    ).astype(int)
    contraction_region_grid, contraction_regions = stable_region_ids(contraction_labels)

    region_info = {}
    for region_id, component in enumerate(contraction_regions, start=1):
        cells = list(component["cells"])
        centroid = np.asarray(cells, dtype=float).mean(axis=0)
        included_classic_regions = sorted({int(classic["region_grid"][cell]) for cell in cells})
        phase_id = int(contraction_labels[cells[0]])
        choices = []
        for classic_region_id in included_classic_regions:
            info = classic["region_info"][classic_region_id]
            record = info["record"]
            location = np.asarray([info["row_index"], info["col_index"]], dtype=float)
            distance = float(np.linalg.norm(location - centroid))
            choices.append((-record.cycle_rank, -record.edges, -record.nodes, distance, classic_region_id, info))
        choices.sort(key=lambda item: item[:5])
        representative_info = choices[0][-1]
        classic_phase_ids = sorted({
            int(classic["region_info"][classic_region_id]["phase_id"])
            for classic_region_id in included_classic_regions
        })
        signatures = sorted({
            compact_signature_text(contraction_data[classic_region_id]["signature"])
            for classic_region_id in included_classic_regions
        })
        region_info[region_id] = {
            "mode": "contraction",
            "mode_title": PHASE_MODE_BY_KEY["contraction"]["title"],
            "phase_id": phase_id,
            "region_id": int(region_id),
            "row_index": int(representative_info["row_index"]),
            "col_index": int(representative_info["col_index"]),
            "record": representative_info["record"],
            "stable_signature": "contraction:" + hashlib.sha256("|".join(signatures).encode()).hexdigest()[:16],
            "classic_region_ids": included_classic_regions,
            "classic_phase_ids": classic_phase_ids,
            "cell_count": int(len(cells)),
        }

    stats = label_component_stats(contraction_labels, min_component_cells=stable_min_component_cells(lambdas, gammas))
    merged_regions = len(classic_region_ids) - len(groups)
    data = {
        "mode": "contraction",
        "mode_title": PHASE_MODE_BY_KEY["contraction"]["title"],
        "labels": contraction_labels,
        "region_grid": contraction_region_grid,
        "regions": contraction_regions,
        "region_info": region_info,
        "classic": classic,
        "classic_region_to_phase_id": classic_region_to_phase_id,
        "accessible_pairs": accessible_pairs,
        "audit": {
            "min_component_cells": int(stable_min_component_cells(lambdas, gammas)),
            "raw_classes": classic["audit"]["raw_classes"],
            "classic_stable_classes": classic["audit"]["stable_classes"],
            "classic_stable_components": classic["audit"]["stable_components"],
            "mode_classes": stats["classes"],
            "mode_components": stats["components"],
            "mode_small_components": stats["small_components"],
            "mode_small_cells": stats["small_cells"],
            "contraction_accessible_pairs": len(accessible_pairs),
            "contraction_merged_classic_regions": int(merged_regions),
            "stable_classes": stats["classes"],
            "stable_components": stats["components"],
            "stable_small_components": stats["small_components"],
            "stable_small_cells": stats["small_cells"],
            "reassigned_cells": 0,
        },
    }
    _PHASE_MODE_CACHE[cache_key] = data
    return data


def phase_mode_data(records, spec, mode_key: str = "classic", *, lambdas=PHASE_LAMBDAS, gammas=None) -> dict:
    if mode_key == "classic":
        return build_classic_mode_data(records, spec, lambdas=lambdas, gammas=gammas)
    if mode_key == "contraction":
        return build_contraction_mode_data(records, spec, lambdas=lambdas, gammas=gammas)
    raise ValueError(f"unknown phase mode: {mode_key}")


In [ ]:
configure_reference_style()

_PHASE_MODE_CACHE.clear()
all_records = []
all_records_by_transition = {}
transition_gammas = {}
cache_paths = []
for spec in TRANSITIONS:
    records, cache_path = cached_transition_scan(spec)
    gammas = gammas_from_records(records)
    all_records_by_transition[spec["key"]] = records
    transition_gammas[spec["key"]] = gammas
    all_records.extend(records)
    cache_paths.append(cache_path)

print("total displayed cells:", len(all_records))
print("sources:", dict(Counter(record.source for record in all_records)))
print("errors:", sum(record.error is not None for record in all_records))
print("terminal Gamma by transition:", {key: round(float(values[-1]), 6) for key, values in transition_gammas.items()})


## Audit the records before plotting

The scan summary must be read before the figures. Check the number of evaluated
cells, data source used for each cell, explicit errors, terminal Gamma retained
for each transition, and endpoint distinctions. The following helper cells write
stable CSV tables and verify that leaves, failed cells, or unstable small regions
have not been hidden by plotting logic.


### Small I/O helpers

Local CSV helper used by phase, audit, legend, and interactive-geometry outputs.


In [ ]:
def write_csv(path, rows):
    """Write a list of dictionaries to CSV with a stable union of field names."""
    from pathlib import Path
    import csv

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    rows = list(rows)
    if not rows:
        path.write_text("", encoding="utf-8")
        return path

    fieldnames = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                fieldnames.append(key)
                seen.add(key)

    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

    return path

def update_table_csv(path, table_name, rows):
    """Store one logical table inside a combined CSV report.

    Existing tables with other names are preserved; rerunning a cell replaces only
    the matching table. This keeps application outputs compact while preserving
    the original row dictionaries.
    """
    from pathlib import Path
    import csv

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = [dict(row) for row in rows]

    existing = []
    if path.exists() and path.stat().st_size:
        with path.open("r", newline="", encoding="utf-8") as handle:
            existing = [
                dict(row)
                for row in csv.DictReader(handle)
                if row.get("table") != table_name
            ]

    combined = existing + [{"table": table_name, **row} for row in rows]
    if not combined:
        path.write_text("", encoding="utf-8")
        return path

    fieldnames = ["table"]
    seen = {"table"}
    for row in combined:
        for key in row:
            if key not in seen:
                fieldnames.append(key)
                seen.add(key)

    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(combined)
    tmp.replace(path)
    return path



In [ ]:
def audit_transition(records, spec, gammas):
    records = list(records)
    gammas = np.asarray(gammas, dtype=float)
    error_records = [record for record in records if record.error or record.source == "error"]
    leaf_records = [record for record in records if any(degree == 1 for degree in record.degree_sequence)]
    by_gamma_lambda = {(round(record.gamma, 10), round(record.lam, 10)): record for record in records}
    endpoint_same_gammas = []
    endpoint_same_reference_gammas = []
    low_gamma_value = float(gammas[0])
    low_left = by_gamma_lambda[(round(low_gamma_value, 10), round(float(PHASE_LAMBDAS[0]), 10))]
    low_right = by_gamma_lambda[(round(low_gamma_value, 10), round(float(PHASE_LAMBDAS[-1]), 10))]
    low_gamma_distinct = low_left.phase_signature != low_right.phase_signature
    for gamma in gammas:
        left = by_gamma_lambda[(round(float(gamma), 10), round(float(PHASE_LAMBDAS[0]), 10))]
        right = by_gamma_lambda[(round(float(gamma), 10), round(float(PHASE_LAMBDAS[-1]), 10))]
        if left.phase_signature == right.phase_signature:
            gamma_value = float(gamma)
            endpoint_same_gammas.append(gamma_value)
            if gamma_value <= ENDPOINT_DISTINCTION_GAMMA_MAX + 1e-12:
                endpoint_same_reference_gammas.append(gamma_value)
    sources = Counter(record.source for record in records)
    row = {
        "transition": spec["key"],
        "title": spec["title"],
        "lambda_samples": len(PHASE_LAMBDAS),
        "gamma_samples": len(gammas),
        "candidate_gamma_samples": len(PHASE_GAMMAS),
        "dimension": SKELETON_DIMENSION,
        "records": len(records),
        "errors": len(error_records),
        "leaf_records": len(leaf_records),
        "distinct_yamada_phases": len({record.phase_signature for record in records}),
        "endpoint_low_gamma": low_gamma_value,
        "endpoint_distinct_low_gamma": low_gamma_distinct,
        "endpoint_reference_gamma_max": ENDPOINT_DISTINCTION_GAMMA_MAX,
        "endpoint_distinct_reference_band": len(endpoint_same_reference_gammas) == 0,
        "endpoint_same_reference_gammas": json.dumps(endpoint_same_reference_gammas),
        "endpoint_same_reference_count": len(endpoint_same_reference_gammas),
        "endpoint_distinct_all_gammas": len(endpoint_same_gammas) == 0,
        "endpoint_same_gammas": json.dumps(endpoint_same_gammas),
        "endpoint_same_gamma_count": len(endpoint_same_gammas),
        "embedded_records": sources.get("embedded", 0),
        "graph_records": sources.get("graph", 0),
        "graph_fallback_records": sources.get("graph-fallback", 0),
        "vertex_records": sources.get("vertex", 0),
    }
    if error_records:
        raise AssertionError(f"{spec['key']} still has errors: {error_records[:3]}")
    if leaf_records:
        raise AssertionError(f"{spec['key']} still has leaf artifacts: {leaf_records[:3]}")
    if not low_gamma_distinct:
        raise AssertionError(f"{spec['key']} has identical endpoint Yamada signatures already at the low-Gamma baseline Gamma={low_gamma_value}")
    return row


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

summary_rows = []
mode_summary_rows = []
legend_rows = []
mode_legend_rows = []
record_rows = []
stability_rows = []
for spec in TRANSITIONS:
    records = all_records_by_transition[spec["key"]]
    gammas = transition_gammas[spec["key"]]
    raw_labels, raw_signature_to_id = phase_ids_for_records(records, gammas=gammas)
    classic_data = phase_mode_data(records, spec, "classic", gammas=gammas)
    contraction_data = phase_mode_data(records, spec, "contraction", gammas=gammas)
    stable_labels = classic_data["labels"]
    stable_raw_ids = classic_data["stable_raw_ids"]
    reassigned_mask = classic_data["reassigned_mask"]
    stable_region_grid = classic_data["region_grid"]
    stable_signature_to_id = classic_data["stable_signature_to_id"]
    stability_audit = classic_data["audit"]
    id_to_raw_signature = {phase_id: signature for signature, phase_id in raw_signature_to_id.items()}
    vertex_signatures = {
        record.phase_signature
        for record in records
        if is_one_vertex_yamada_record(record)
    }
    top_gamma_index = len(gammas) - 1
    top_gamma_value = float(gammas[top_gamma_index])
    top_gamma_records = [record for record in records if round(float(record.gamma), 10) == round(top_gamma_value, 10)]
    top_gamma_vertex_records = [record for record in top_gamma_records if is_one_vertex_yamada_record(record)]
    top_gamma_vertex_lambdas = [round(float(record.lam), 6) for record in top_gamma_vertex_records]
    top_gamma_classic_vertex_cells = sum(
        id_to_raw_signature[int(stable_raw_ids[top_gamma_index, col_index])] in vertex_signatures
        for col_index in range(stable_raw_ids.shape[1])
    )
    if len(top_gamma_vertex_records) != len(PHASE_LAMBDAS):
        raise AssertionError(
            f"{spec['key']} terminal Gamma={top_gamma_value} is not an all-lambda raw one-vertex row"
        )
    if top_gamma_classic_vertex_cells != len(PHASE_LAMBDAS):
        raise AssertionError(
            f"{spec['key']} terminal Gamma={top_gamma_value} is not an all-lambda Classic one-vertex row"
        )
    summary_row = audit_transition(records, spec, gammas)
    summary_row.update({key: stability_audit[key] for key in [
        "min_component_cells", "raw_classes", "raw_components", "raw_small_components",
        "raw_small_cells", "stable_classes", "stable_components", "stable_small_components",
        "stable_small_cells", "reassigned_cells",
    ]})
    summary_row.update({
        "contraction_classes": contraction_data["audit"]["mode_classes"],
        "contraction_components": contraction_data["audit"]["mode_components"],
        "contraction_small_components": contraction_data["audit"]["mode_small_components"],
        "contraction_accessible_pairs": contraction_data["audit"]["contraction_accessible_pairs"],
        "contraction_merged_classic_regions": contraction_data["audit"]["contraction_merged_classic_regions"],
        "top_gamma": top_gamma_value,
        "top_gamma_is_terminal_vertex_row": True,
        "top_gamma_raw_vertex_cells": len(top_gamma_vertex_records),
        "top_gamma_classic_vertex_cells": int(top_gamma_classic_vertex_cells),
        "top_gamma_vertex_lambda_min": min(top_gamma_vertex_lambdas) if top_gamma_vertex_lambdas else "",
        "top_gamma_vertex_lambda_max": max(top_gamma_vertex_lambdas) if top_gamma_vertex_lambdas else "",
    })
    summary_rows.append(summary_row)
    stability_rows.append({"transition": spec["key"], "title": spec["title"], **stability_audit})

    for mode in PHASE_MODES:
        data = phase_mode_data(records, spec, mode["key"], gammas=gammas)
        audit = data["audit"]
        mode_summary_rows.append({
            "transition": spec["key"],
            "title": spec["title"],
            "mode": mode["key"],
            "mode_title": mode["title"],
            "lambda_samples": len(PHASE_LAMBDAS),
            "gamma_samples": len(gammas),
            "candidate_gamma_samples": len(PHASE_GAMMAS),
            "dimension": SKELETON_DIMENSION,
            "records": len(records),
            "raw_yamada_classes": len({record.phase_signature for record in records}),
            "mode_classes": audit["mode_classes"],
            "mode_components": audit["mode_components"],
            "mode_small_components": audit["mode_small_components"],
            "mode_small_cells": audit["mode_small_cells"],
            "classic_stable_components": classic_data["audit"]["stable_components"],
            "contraction_accessible_pairs": audit["contraction_accessible_pairs"],
            "contraction_merged_classic_regions": audit["contraction_merged_classic_regions"],
            "top_gamma": top_gamma_value,
            "top_gamma_is_terminal_vertex_row": True,
            "top_gamma_raw_vertex_cells": len(top_gamma_vertex_records),
            "top_gamma_classic_vertex_cells": int(top_gamma_classic_vertex_cells),
            "top_gamma_vertex_lambda_min": min(top_gamma_vertex_lambdas) if top_gamma_vertex_lambdas else "",
            "top_gamma_vertex_lambda_max": max(top_gamma_vertex_lambdas) if top_gamma_vertex_lambdas else "",
        })

    by_signature = {}
    for record in records:
        by_signature.setdefault(record.phase_signature, record)
    for signature, stable_phase_id in stable_signature_to_id.items():
        record = by_signature[signature]
        legend_rows.append({
            "transition": spec["key"],
            "stable_phase_id": stable_phase_id,
            "raw_phase_id": raw_signature_to_id[signature],
            "source_first_seen": record.source,
            "polynomial": record.polynomial,
            "phase_signature": signature,
        })

    for mode in PHASE_MODES:
        data = phase_mode_data(records, spec, mode["key"], gammas=gammas)
        labels = data["labels"]
        for phase_id in sorted(set(int(value) for value in labels.ravel())):
            region_ids = sorted(
                region_id
                for region_id, info in data["region_info"].items()
                if int(info["phase_id"]) == phase_id
            )
            representative_info = data["region_info"][region_ids[0]]
            record = representative_info["record"]
            mode_legend_rows.append({
                "transition": spec["key"],
                "title": spec["title"],
                "mode": mode["key"],
                "mode_title": mode["title"],
                "phase_id": phase_id,
                "connected_region_ids": json.dumps(region_ids),
                "classic_region_ids": json.dumps(sorted({rid for region_id in region_ids for rid in data["region_info"][region_id]["classic_region_ids"]})),
                "classic_phase_ids": json.dumps(sorted({pid for region_id in region_ids for pid in data["region_info"][region_id]["classic_phase_ids"]})),
                "representative_lambda": round(float(record.lam), 6),
                "representative_Gamma": round(float(record.gamma), 6),
                "representative_source": record.source,
                "representative_core_mode": record.core_mode,
                "representative_polynomial": record.polynomial,
                "phase_signature": representative_info["stable_signature"],
            })

    lambda_index = {round(float(value), 10): index for index, value in enumerate(PHASE_LAMBDAS)}
    gamma_index = {round(float(value), 10): index for index, value in enumerate(gammas)}
    for record in records:
        row_index = gamma_index[round(float(record.gamma), 10)]
        col_index = lambda_index[round(float(record.lam), 10)]
        raw_phase_id = int(raw_labels[row_index, col_index])
        stable_raw_phase_id = int(stable_raw_ids[row_index, col_index])
        stable_signature = id_to_raw_signature[stable_raw_phase_id]
        stable_phase_id = int(stable_labels[row_index, col_index])
        contraction_phase_id = int(contraction_data["labels"][row_index, col_index])
        contraction_region_id = int(contraction_data["region_grid"][row_index, col_index])
        record_rows.append({
            "transition": record.transition,
            "title": record.title,
            "lambda": record.lam,
            "Gamma": record.gamma,
            "raw_phase_id": raw_phase_id,
            "stable_phase_id": stable_phase_id,
            "stable_region_id": int(stable_region_grid[row_index, col_index]),
            "contraction_phase_id": contraction_phase_id,
            "contraction_region_id": contraction_region_id,
            "stable_reassigned": bool(reassigned_mask[row_index, col_index]),
            "source": record.source,
            "core_mode": record.core_mode,
            "nodes": record.nodes,
            "edges": record.edges,
            "components": record.components,
            "cycle_rank": record.cycle_rank,
            "degree_sequence": repr(record.degree_sequence),
            "phase_signature": record.phase_signature,
            "stable_phase_signature": stable_signature,
            "polynomial": record.polynomial,
            "error": record.error or "",
        })

report_path = PHASE_REPORT_CSV
update_table_csv(report_path, "audit_summary", summary_rows)
update_table_csv(report_path, "mode_summary", mode_summary_rows)
update_table_csv(report_path, "legend", legend_rows)
update_table_csv(report_path, "mode_legend", mode_legend_rows)
update_table_csv(report_path, "records", record_rows)
update_table_csv(report_path, "stability_audit", stability_rows)

print("audit summary:")
for row in summary_rows:
    print(row)
print("mode summary:")
for row in mode_summary_rows:
    print(row)
print("saved:", report_path)


## Static phase figures and two classifications

The **classic** map labels stabilized connected regions by their exact Yamada
signature. The **up to contraction moves** map additionally groups selected
representative cores under the notebook's stated contraction convention. These
are different questions: a merged contraction class does not assert literal
equality of all classic signatures.

Read phase boundaries as nearest-neighbor boundaries on the sampled grid. A
single-cell island, a boundary-touching surface, or a cell produced only after a
fallback requires inspection rather than automatic physical interpretation.


In [ ]:
# ============================================================
# REUSABLE PAPER FIGURE STYLE
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt


def configure_paper_style():
    """
    Global Matplotlib style used for publication figures.
    Call once near the beginning of the notebook/script.
    """

    mpl.rcParams.update({

        # ----------------------------------------------------
        # Typography
        # ----------------------------------------------------
        "font.family": "serif",
        "font.serif": [
            "Times New Roman",
            "Times",
            "DejaVu Serif",
        ],
        "mathtext.fontset": "dejavuserif",

        # ----------------------------------------------------
        # Figure / export
        # ----------------------------------------------------
        "figure.dpi": 220,
        "savefig.dpi": 450,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "figure.facecolor": "white",
        "axes.facecolor": "white",

        # ----------------------------------------------------
        # Axes
        # ----------------------------------------------------
        "axes.linewidth": 1.5,
        "axes.edgecolor": "black",

        # ----------------------------------------------------
        # Tick directions
        # ----------------------------------------------------
        "xtick.direction": "out",
        "ytick.direction": "out",

        # ----------------------------------------------------
        # PDF / vector compatibility
        # ----------------------------------------------------
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })
    mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],

    # Force Computer Modern math symbols
    "mathtext.fontset": "cm",

    # IMPORTANT: keep Matplotlib mathtext active
    "text.usetex": False,

    "figure.dpi": 220,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,

    "axes.linewidth": 1.5,

    "xtick.direction": "out",
    "ytick.direction": "out",
    })

def style_paper_axis(
    ax,
    xlabel=None,
    ylabel=None,
    title=None,
    title_size=24,
    label_size=24,
    tick_size=18,
    title_weight="semibold",
):
    """
    Apply the common paper styling to one Matplotlib axis.
    """

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------
    if xlabel is not None:
        ax.set_xlabel(
            xlabel,
            fontsize=label_size,
            labelpad=7,
        )

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            fontsize=label_size,
            labelpad=8,
        )

    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------
    if title is not None:
        ax.set_title(
            title,
            fontsize=title_size,
            fontweight=title_weight,
            pad=14,
        )

    # --------------------------------------------------------
    # Major ticks
    # --------------------------------------------------------
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=tick_size,
        width=1.4,
        length=6,
        direction="out",
    )

    # --------------------------------------------------------
    # Minor ticks
    # --------------------------------------------------------
    ax.minorticks_on()

    ax.tick_params(
        axis="both",
        which="minor",
        width=1.0,
        length=3,
        direction="out",
    )

    # --------------------------------------------------------
    # Spines
    # --------------------------------------------------------
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color("black")


def style_unlabeled_colorbar(cbar):
    """
    Paper-style discrete colorbar with no text/tick labels.
    """

    cbar.set_ticks([])
    cbar.ax.set_yticks([])
    cbar.ax.set_yticklabels([])

    cbar.ax.tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelright=False,
        length=0,
    )

    cbar.outline.set_linewidth(1.2)
    cbar.outline.set_edgecolor("black")

    if hasattr(cbar, "dividers"):
        cbar.dividers.set_linewidth(0.8)
        cbar.dividers.set_color("black")


configure_paper_style()


        
OKABE_ITO_5 = ["#ffaa0e", "#d62728", "#1f77b4", "#9467bd", "#2ca02c"]


def categorical_cmap(class_count: int):
    if class_count <= len(OKABE_ITO_5):
        colors = OKABE_ITO_5[:class_count]
    elif class_count <= 20:
        colors = [mpl.colors.to_hex(plt.get_cmap("tab20")(i)) for i in range(class_count)]
    else:
        colors = [mpl.colors.to_hex(plt.get_cmap("turbo")(value)) for value in np.linspace(0.04, 0.96, class_count)]
    return ListedColormap(colors, name=f"yamada_classes_{class_count}")


def draw_phase_boundaries(ax, lambdas, gammas, labels):
    for phase_id in sorted(set(int(value) for value in labels.ravel())):
        mask = (labels == phase_id).astype(float)
        if mask.min() == mask.max():
            continue
        ax.contour(
            lambdas,
            gammas,
            mask,
            levels=[0.5],
            colors="black",
            linewidths=1.0,
            alpha=1.0,
        )


def phase_map_title(spec, mode_key="classic"):
    start_name = str(spec.get("start_name", "H_0")).replace(" ", r"\;")
    end_name = str(spec.get("end_name", "H_1")).replace(" ", r"\;")
    title = rf"$(1-\lambda)H_{{\mathrm{{{start_name}}}}}+\lambda H_{{\mathrm{{{end_name}}}}}$"
    if mode_key != "classic":
        title = title + rf" ({PHASE_MODE_BY_KEY[mode_key]['short_title']})"
    return title


def plot_reference_style_phase_map(records, spec, *, out_stem=None, ax=None, show_colorbar=True, mode_key="classic", gammas=None):
    gammas = gammas_from_records(records) if gammas is None else np.asarray(gammas, dtype=float)
    mode = PHASE_MODE_BY_KEY[mode_key]
    mode_data = phase_mode_data(records, spec, mode_key, gammas=gammas)
    labels = mode_data["labels"]
    audit = mode_data["audit"]
    class_count = int(labels.max())
    cmap = categorical_cmap(class_count)
    norm = BoundaryNorm(np.arange(0.5, class_count + 1.5, 1.0), cmap.N)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(8.6, 6.1), facecolor="white")
    else:
        fig = ax.figure
    ax.set_facecolor("white")

    image = ax.pcolormesh(
        PHASE_LAMBDAS,
        gammas,
        labels,
        shading="nearest",
        cmap=cmap,
        norm=norm,
        rasterized=False,
    )
    draw_phase_boundaries(ax, PHASE_LAMBDAS, gammas, labels)
    ax.set_xlim(float(PHASE_LAMBDAS[0]), float(PHASE_LAMBDAS[-1]))
    ax.set_ylim(float(gammas[0]), float(gammas[-1]))
    style_paper_axis(
        ax,
        xlabel=r"$\lambda$",
        ylabel=r"$E$",
        title=phase_map_title(spec, mode_key),
        title_size=24,
        label_size=24,
        tick_size=18,
        title_weight="semibold",
    )

    if show_colorbar:
        cbar = fig.colorbar(
            image,
            ax=ax,
            boundaries=np.arange(0.5, class_count + 1.5, 1.0),
            spacing="uniform",
            fraction=0.046,
            pad=0.035,
            drawedges=True,
        )
        cbar.ax.set_title(r"$\Upsilon$", fontsize=22, fontweight="semibold", pad=10)
        style_unlabeled_colorbar(cbar)

    if out_stem:
        png_path = FIGURE_DIR / f"{out_stem}.png"
        pdf_path = FIGURE_DIR / f"{out_stem}.pdf"
        fig.savefig(png_path, dpi=450, bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="none")
        fig.savefig(pdf_path, dpi=700, bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="none")
        print("saved:", png_path)
        print("saved:", pdf_path)
    return fig, ax, mode_data, audit


individual_outputs = []
for spec in TRANSITIONS:
    records = all_records_by_transition[spec["key"]]
    gammas = transition_gammas[spec["key"]]
    for mode in PHASE_MODES:
        if mode["key"] == "classic":
            out_stem = f"06_hamiltonian_yamada_phase_map_{spec['key']}"
        else:
            out_stem = f"06_hamiltonian_yamada_phase_map_up_to_contraction_{spec['key']}"
        fig, ax, mode_data, audit = plot_reference_style_phase_map(
            records,
            spec,
            out_stem=out_stem,
            mode_key=mode["key"],
            gammas=gammas,
        )
        individual_outputs.append((spec["key"], mode["key"], audit["mode_classes"], audit["mode_components"], round(float(gammas[-1]), 6)))
        plt.show()

print("individual phase counts:", individual_outputs)


In [ ]:
def plot_five_transition_overview(mode_key="classic", *, filename_suffix=""):
    mode = PHASE_MODE_BY_KEY[mode_key]
    fig, axes = plt.subplots(
        len(TRANSITIONS), 1,
        figsize=(8.4, 2.35 * len(TRANSITIONS)),
        sharex=True,
        constrained_layout=True,
        facecolor="white",
    )
    axes = np.atleast_1d(axes)
    for ax, spec in zip(axes, TRANSITIONS):
        records = all_records_by_transition[spec["key"]]
        gammas = transition_gammas[spec["key"]]
        mode_data = phase_mode_data(records, spec, mode_key, gammas=gammas)
        labels = mode_data["labels"]
        audit = mode_data["audit"]
        class_count = int(labels.max())
        cmap = categorical_cmap(class_count)
        norm = BoundaryNorm(np.arange(0.5, class_count + 1.5, 1.0), cmap.N)
        ax.set_facecolor("white")
        ax.pcolormesh(PHASE_LAMBDAS, gammas, labels, shading="nearest", cmap=cmap, norm=norm, rasterized=False)
        draw_phase_boundaries(ax, PHASE_LAMBDAS, gammas, labels)
        ax.set_xlim(float(PHASE_LAMBDAS[0]), float(PHASE_LAMBDAS[-1]))
        ax.set_ylim(float(gammas[0]), float(gammas[-1]))
        style_paper_axis(
            ax,
            ylabel=r"$E$",
            title=phase_map_title(spec, mode_key),
            title_size=16,
            label_size=18,
            tick_size=12,
            title_weight="semibold",
        )
    axes[-1].set_xlabel(r"$\lambda$", fontsize=18, labelpad=7)
    fig.suptitle(f"Five Hamiltonian phase partitions - {mode['title']}", fontsize=20, fontweight="semibold")
    png_path = FIGURE_DIR / f"06_hamiltonian_yamada_phase_maps_5transition_overview{filename_suffix}.png"
    pdf_path = FIGURE_DIR / f"06_hamiltonian_yamada_phase_maps_5transition_overview{filename_suffix}.pdf"
    fig.savefig(png_path, dpi=450, bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="none")
    fig.savefig(pdf_path, dpi=700, bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="none")
    print("saved:", png_path)
    print("saved:", pdf_path)
    return fig


def plot_two_mode_comparison_overview():
    fig, axes = plt.subplots(
        len(TRANSITIONS), len(PHASE_MODES),
        figsize=(13.6, 2.25 * len(TRANSITIONS)),
        sharex=True,
        sharey=False,
        constrained_layout=True,
        facecolor="white",
    )
    for row_index, spec in enumerate(TRANSITIONS):
        records = all_records_by_transition[spec["key"]]
        gammas = transition_gammas[spec["key"]]
        for col_index, mode in enumerate(PHASE_MODES):
            ax = axes[row_index, col_index]
            mode_data = phase_mode_data(records, spec, mode["key"], gammas=gammas)
            labels = mode_data["labels"]
            audit = mode_data["audit"]
            class_count = int(labels.max())
            cmap = categorical_cmap(class_count)
            norm = BoundaryNorm(np.arange(0.5, class_count + 1.5, 1.0), cmap.N)
            ax.set_facecolor("white")
            ax.pcolormesh(PHASE_LAMBDAS, gammas, labels, shading="nearest", cmap=cmap, norm=norm, rasterized=False)
            draw_phase_boundaries(ax, PHASE_LAMBDAS, gammas, labels)
            ax.set_xlim(float(PHASE_LAMBDAS[0]), float(PHASE_LAMBDAS[-1]))
            ax.set_ylim(float(gammas[0]), float(gammas[-1]))
            ylabel = r"$E$" if col_index == 0 else None
            xlabel = r"$\lambda$" if row_index == len(TRANSITIONS) - 1 else None
            title = phase_map_title(spec, mode["key"]) if col_index == 0 else mode["title"]
            style_paper_axis(
                ax,
                xlabel=xlabel,
                ylabel=ylabel,
                title=title,
                title_size=13,
                label_size=16,
                tick_size=11,
                title_weight="semibold",
            )
    fig.suptitle("Classic Yamada phases versus phases up to contraction moves", fontsize=20, fontweight="semibold")
    png_path = FIGURE_DIR / "06_hamiltonian_yamada_phase_maps_5transition_two_mode_comparison.png"
    pdf_path = FIGURE_DIR / "06_hamiltonian_yamada_phase_maps_5transition_two_mode_comparison.pdf"
    fig.savefig(png_path, dpi=450, bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="none")
    fig.savefig(pdf_path, dpi=700, bbox_inches="tight", pad_inches=0.03, facecolor="white", edgecolor="none")
    print("saved:", png_path)
    print("saved:", pdf_path)
    return fig


fig = plot_five_transition_overview("classic", filename_suffix="")
plt.show()
fig = plot_five_transition_overview("contraction", filename_suffix="_up_to_contraction")
plt.show()
fig = plot_two_mode_comparison_overview()
plt.show()


In [ ]:
# Compact endpoint check for the exact issue that can make endpoint columns look falsely identical.
endpoint_rows = []
for spec in TRANSITIONS:
    records = all_records_by_transition[spec["key"]]
    gammas = transition_gammas[spec["key"]]
    classic_data = phase_mode_data(records, spec, "classic", gammas=gammas)
    contraction_data = phase_mode_data(records, spec, "contraction", gammas=gammas)
    stable_raw_ids = classic_data["stable_raw_ids"]
    raw_signature_to_id = classic_data["raw_signature_to_id"]
    id_to_raw_signature = {phase_id: signature for signature, phase_id in raw_signature_to_id.items()}
    by_gamma_lambda = {(round(record.gamma, 10), round(record.lam, 10)): record for record in records}
    for row_index, gamma in enumerate(gammas):
        left = by_gamma_lambda[(round(float(gamma), 10), round(float(PHASE_LAMBDAS[0]), 10))]
        right = by_gamma_lambda[(round(float(gamma), 10), round(float(PHASE_LAMBDAS[-1]), 10))]
        stable_left = id_to_raw_signature[int(stable_raw_ids[row_index, 0])]
        stable_right = id_to_raw_signature[int(stable_raw_ids[row_index, -1])]
        contraction_left = int(contraction_data["labels"][row_index, 0])
        contraction_right = int(contraction_data["labels"][row_index, -1])
        endpoint_rows.append({
            "transition": spec["key"],
            "Gamma": float(gamma),
            "terminal_Gamma": float(gammas[-1]),
            "in_endpoint_reference_band": float(gamma) <= ENDPOINT_DISTINCTION_GAMMA_MAX + 1e-12,
            "raw_lambda_0_phase": left.phase_signature,
            "raw_lambda_1_phase": right.phase_signature,
            "raw_distinct": left.phase_signature != right.phase_signature,
            "classic_lambda_0_phase": stable_left,
            "classic_lambda_1_phase": stable_right,
            "classic_distinct": stable_left != stable_right,
            "contraction_lambda_0_phase": contraction_left,
            "contraction_lambda_1_phase": contraction_right,
            "contraction_distinct": contraction_left != contraction_right,
        })

endpoint_path = PHASE_REPORT_CSV
update_table_csv(endpoint_path, "endpoint_distinction", endpoint_rows)
baseline_gamma = float(PHASE_GAMMAS[0])
baseline_endpoint_rows = [row for row in endpoint_rows if abs(row["Gamma"] - baseline_gamma) < 1e-12]
# Endpoint correctness check should assert raw Yamada only
if not all(row["raw_distinct"] for row in baseline_endpoint_rows):
    failures = [row for row in baseline_endpoint_rows if not row["raw_distinct"]]
    raise AssertionError(f"Raw endpoint Yamada signatures matched unexpectedly: {failures[:5]}")
classic_matches = [row for row in endpoint_rows if not row["classic_distinct"]]
print(f"lambda=0 and lambda=1 raw/Classic endpoint Yamada signatures are distinct at the low-Gamma baseline Gamma={baseline_gamma}")
print(f"recorded {len(classic_matches)} Classic endpoint coincidences elsewhere in the displayed grid; these remain visible in the combined CSV report")
print("the final Gamma row for each transition is the first row where all lambda cells are one-vertex Yamada")
print("contraction endpoint equality is reported but not asserted because this mode intentionally merges contraction-accessible graphs")
print("saved:", endpoint_path)


## Interactive Plotly geometry QA

The stable two-dimensional phase partitions are clickable. Select a mode and a
transition, then click a region to load one representative exceptional surface
and the corresponding simplified Yamada skeleton.

Use the view as a geometry audit, not only as decoration: confirm that the
surface, black skeleton, red graph vertices, component counts, cycle rank, and
displayed polynomial describe the same representative record. The HTML is a
self-contained data snapshot except for its pinned Plotly CDN script.


In [ ]:
QA_GEOMETRY_VERSION = "hamiltonian_yamada_plotly_region_geometry_v15_application_results"
QA_DATA_PATH = RESULT_DIR / "06_hamiltonian_yamada_plotly_region_geometry.json"
QA_REPRESENTATIVES_PATH = PHASE_REPORT_CSV
QA_HTML_PATH = FIGURE_DIR / "06_hamiltonian_yamada_plotly_region_geometry.html"
QA_MAX_SURFACE_TRIANGLES = 1200
QA_MAX_EDGE_POINTS = 120


def interactive_specs() -> list[dict]:
    return list(TRANSITIONS)


def interactive_records_for_spec(spec) -> list[HamiltonianPhaseRecord]:
    return all_records_by_transition[spec["key"]]


def interactive_lambdas_for_spec(spec) -> np.ndarray:
    return np.asarray(PHASE_LAMBDAS, dtype=float)


def interactive_parameters_for_spec(spec) -> np.ndarray:
    return np.asarray(transition_gammas[spec["key"]], dtype=float)


def parameter_name_for_spec(spec) -> str:
    return str(spec.get("parameter_name", "Gamma"))


def parameter_label_for_spec(spec) -> str:
    return str(spec.get("parameter_label", "E"))


def qa_cache_key() -> str:
    specs = interactive_specs()
    payload = {
        "version": QA_GEOMETRY_VERSION,
        "transition_keys": [spec["key"] for spec in specs],
        "mode_keys": [mode["key"] for mode in PHASE_MODES],
        "lambdas_by_transition": {
            spec["key"]: [round(float(value), 10) for value in interactive_lambdas_for_spec(spec)]
            for spec in specs
        },
        "parameters_by_transition": {
            spec["key"]: [round(float(value), 10) for value in interactive_parameters_for_spec(spec)]
            for spec in specs
        },
        "parameter_names": {spec["key"]: parameter_name_for_spec(spec) for spec in specs},
        "dimension": {spec["key"]: int(spec.get("dimension", SKELETON_DIMENSION)) for spec in specs},
        "stable_min_component_cells": {
            spec["key"]: stable_min_component_cells(interactive_lambdas_for_spec(spec), interactive_parameters_for_spec(spec))
            for spec in specs
        },
        "stable_min_component_fraction": float(STABLE_MIN_COMPONENT_FRACTION),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


def safe_float_list(values, *, ndigits: int = 5) -> list[float]:
    return [round(float(value), ndigits) for value in values]


def empty_surface_payload() -> dict:
    return {
        "x": [], "y": [], "z": [], "i": [], "j": [], "k": [],
        "point_count": 0, "triangle_count": 0,
    }


def mesh_to_plotly_payload(polydata, *, max_triangles: int = QA_MAX_SURFACE_TRIANGLES, allow_empty: bool = False) -> dict:
    mesh = polydata.extract_surface().triangulate().clean()
    if mesh.n_points == 0 or mesh.n_cells == 0:
        if allow_empty:
            return empty_surface_payload()
        raise ValueError("representative exceptional surface is empty")
    if mesh.n_cells > max_triangles:
        reduction = max(0.0, min(0.92, 1.0 - max_triangles / max(int(mesh.n_cells), 1)))
        try:
            mesh = mesh.decimate_pro(reduction, preserve_topology=True).triangulate().clean()
        except Exception:
            mesh = mesh.triangulate().clean()

    faces_flat = np.asarray(mesh.faces, dtype=np.int64)
    faces = []
    cursor = 0
    while cursor < len(faces_flat):
        face_size = int(faces_flat[cursor])
        vertices = faces_flat[cursor + 1 : cursor + 1 + face_size]
        if face_size == 3:
            faces.append(vertices)
        elif face_size > 3:
            for index in range(1, face_size - 1):
                faces.append([vertices[0], vertices[index], vertices[index + 1]])
        cursor += face_size + 1

    faces = np.asarray(faces, dtype=np.int64)
    if faces.size == 0:
        if allow_empty:
            return empty_surface_payload()
        raise ValueError("representative exceptional surface has no triangular faces")

    if len(faces) > max_triangles:
        keep = np.unique(np.linspace(0, len(faces) - 1, max_triangles).astype(int))
        faces = faces[keep]

    used_vertices = np.unique(faces.ravel())
    remap = {int(old_index): int(new_index) for new_index, old_index in enumerate(used_vertices)}
    remapped_faces = np.vectorize(lambda value: remap[int(value)])(faces)
    points = np.asarray(mesh.points, dtype=float)[used_vertices]
    return {
        "x": safe_float_list(points[:, 0]),
        "y": safe_float_list(points[:, 1]),
        "z": safe_float_list(points[:, 2]),
        "i": [int(value) for value in remapped_faces[:, 0]],
        "j": [int(value) for value in remapped_faces[:, 1]],
        "k": [int(value) for value in remapped_faces[:, 2]],
        "point_count": int(points.shape[0]),
        "triangle_count": int(remapped_faces.shape[0]),
    }


def _representative_vertex_coord(skeleton: NodalSkeleton) -> np.ndarray:
    try:
        indices = np.argwhere(skeleton._skeleton_image)
    except Exception as exc:
        if skeleton_failure_forces_vertex(exc):
            return np.asarray(skeleton.origin, dtype=float)
        raise
    spacing = skeleton.spacing * skeleton.axis_scale
    if len(indices):
        return idx_to_coord(indices.mean(axis=0), spacing=spacing, origin=skeleton.origin)
    return np.asarray(skeleton.origin, dtype=float)


def graph_to_plotly_payload(graph: nx.MultiGraph, skeleton: NodalSkeleton) -> dict:
    spacing = skeleton.spacing * skeleton.axis_scale
    origin = skeleton.origin
    line_x, line_y, line_z = [], [], []
    node_coords = []

    for _, data in graph.nodes(data=True):
        if "pos" in data:
            node_coords.append(idx_to_coord(np.asarray(data["pos"], dtype=float), spacing=spacing, origin=origin))

    for u, v, _, data in graph.edges(keys=True, data=True):
        points = data.get("pts")
        if points is None:
            points = np.vstack([graph.nodes[u]["pos"], graph.nodes[v]["pos"]])
        points = np.asarray(points, dtype=float)
        if len(points) > QA_MAX_EDGE_POINTS:
            keep = np.unique(np.linspace(0, len(points) - 1, QA_MAX_EDGE_POINTS).astype(int))
            points = points[keep]
        coords = idx_to_coord(points, spacing=spacing, origin=origin)
        line_x.extend(safe_float_list(coords[:, 0]) + [None])
        line_y.extend(safe_float_list(coords[:, 1]) + [None])
        line_z.extend(safe_float_list(coords[:, 2]) + [None])

    if not node_coords:
        node_coords = [_representative_vertex_coord(skeleton)]

    node_array = np.asarray(node_coords, dtype=float)
    return {
        "lines": {"x": line_x, "y": line_y, "z": line_z},
        "nodes": {
            "x": safe_float_list(node_array[:, 0]),
            "y": safe_float_list(node_array[:, 1]),
            "z": safe_float_list(node_array[:, 2]),
        },
        "node_count": int(graph.number_of_nodes()),
        "edge_count": int(graph.number_of_edges()),
    }


def build_region_geometry(spec, record: HamiltonianPhaseRecord, *, color: str) -> dict:
    skeleton = skeleton_for_phase_record(spec, record)
    topology = interior_topology_summary(skeleton)
    graph = one_vertex_graph() if topology["forces_vertex"] else representative_core_graph(skeleton, record)
    try:
        surface_payload = mesh_to_plotly_payload(skeleton.exceptional_surface_pv, allow_empty=graph.number_of_edges() == 0)
    except Exception as exc:
        if graph.number_of_edges() == 0 and skeleton_failure_forces_vertex(exc):
            surface_payload = empty_surface_payload()
        else:
            raise
    skeleton_payload = graph_to_plotly_payload(graph, skeleton)
    if not skeleton_payload["node_count"]:
        raise ValueError(f"{spec['key']} region representative has empty skeleton")
    if topology["forces_vertex"] and skeleton_payload["edge_count"]:
        raise AssertionError(f"{spec['key']} closed genus-zero representative was not collapsed to a vertex")
    return {
        "surface": surface_payload,
        "skeleton": skeleton_payload,
        "topology": topology,
        "color": color,
    }


def color_list_for_count(class_count: int) -> list[str]:
    return [mpl.colors.to_hex(color) for color in categorical_cmap(class_count).colors]


def build_interactive_payload() -> tuple[dict, list[dict]]:
    cache_key = qa_cache_key()
    if QA_DATA_PATH.exists():
        cached = json.loads(QA_DATA_PATH.read_text(encoding="utf-8"))
        if cached.get("version") == QA_GEOMETRY_VERSION and cached.get("cache_key") == cache_key:
            print("loaded interactive geometry:", QA_DATA_PATH)
            representatives = cached.get("representatives", [])
            if representatives:
                update_table_csv(QA_REPRESENTATIVES_PATH, "plotly_region_representatives", representatives)
            return cached, representatives

    specs = interactive_specs()
    payload = {
        "version": QA_GEOMETRY_VERSION,
        "cache_key": cache_key,
        "lambdas": safe_float_list(PHASE_LAMBDAS, ndigits=6),
        "candidate_gammas": safe_float_list(PHASE_GAMMAS, ndigits=6),
        "stable_min_component_cells": {
            spec["key"]: stable_min_component_cells(interactive_lambdas_for_spec(spec), interactive_parameters_for_spec(spec))
            for spec in specs
        },
        "modes": [{key: mode[key] for key in ["key", "title", "short_title", "colorbar_title", "description"]} for mode in PHASE_MODES],
        "transitions": [],
        "regions": {},
    }
    representative_rows = []
    geometry_cache = {}

    for spec in specs:
        records = interactive_records_for_spec(spec)
        lambdas = interactive_lambdas_for_spec(spec)
        parameters = interactive_parameters_for_spec(spec)
        parameter_name = parameter_name_for_spec(spec)
        parameter_label = parameter_label_for_spec(spec)
        transition_payload = {
            "key": spec["key"],
            "title": spec["title"],
            "lambdas": safe_float_list(lambdas, ndigits=6),
            "gammas": safe_float_list(parameters, ndigits=6),
            "parameters": safe_float_list(parameters, ndigits=6),
            "parameterName": parameter_name,
            "parameterLabel": parameter_label,
            "terminalGamma": round(float(parameters[-1]), 6) if parameter_name == "Gamma" else None,
            "startName": spec.get("start_name", ""),
            "endName": spec.get("end_name", ""),
            "lambdaSamples": len(lambdas),
            "modes": {},
        }
        for mode in PHASE_MODES:
            mode_key = mode["key"]
            mode_data = phase_mode_data(records, spec, mode_key, lambdas=lambdas, gammas=parameters)
            labels = mode_data["labels"]
            region_grid = mode_data["region_grid"]
            audit = mode_data["audit"]
            phase_colors = color_list_for_count(int(labels.max()))
            region_keys = [
                [f"{mode_key}:{spec['key']}:{int(region_grid[row_index, col_index])}" for col_index in range(region_grid.shape[1])]
                for row_index in range(region_grid.shape[0])
            ]

            transition_payload["modes"][mode_key] = {
                "z": labels.astype(int).tolist(),
                "regionKeys": region_keys,
                "colors": phase_colors,
                "classes": int(audit["mode_classes"]),
                "components": int(audit["mode_components"]),
                "smallComponents": int(audit["mode_small_components"]),
                "rawClasses": int(audit["raw_classes"]),
                "classicStableComponents": int(phase_mode_data(records, spec, "classic", lambdas=lambdas, gammas=parameters)["audit"]["stable_components"]),
                "contractionAccessiblePairs": int(audit["contraction_accessible_pairs"]),
                "contractionMergedClassicRegions": int(audit["contraction_merged_classic_regions"]),
            }

            for region_id, component in enumerate(mode_data["regions"], start=1):
                info = mode_data["region_info"][region_id]
                record = info["record"]
                phase_id = int(info["phase_id"])
                region_key = f"{mode_key}:{spec['key']}:{region_id}"
                color = phase_colors[phase_id - 1]
                geometry_key = (spec["key"], round(float(record.gamma), 10), round(float(record.lam), 10), record.core_mode, color)
                if geometry_key not in geometry_cache:
                    geometry_cache[geometry_key] = build_region_geometry(spec, record, color=color)
                geometry = geometry_cache[geometry_key]
                region_payload = {
                    "key": region_key,
                    "mode": mode_key,
                    "modeTitle": mode["title"],
                    "transition": spec["key"],
                    "title": spec["title"],
                    "regionId": int(region_id),
                    "phaseId": phase_id,
                    "stableRegionId": int(region_id),
                    "stablePhaseId": phase_id,
                    "stableSignature": info["stable_signature"],
                    "classicRegionIds": [int(value) for value in info["classic_region_ids"]],
                    "classicPhaseIds": [int(value) for value in info["classic_phase_ids"]],
                    "lambda": round(float(record.lam), 6),
                    "Gamma": round(float(record.gamma), 6),
                    "parameterValue": round(float(record.gamma), 6),
                    "parameterName": parameter_name,
                    "parameterLabel": parameter_label,
                    "source": record.source,
                    "coreMode": record.core_mode,
                    "startName": spec.get("start_name", ""),
                    "endName": spec.get("end_name", ""),
                                                                            "nodes": int(record.nodes),
                    "edges": int(record.edges),
                    "components": int(record.components),
                    "cycleRank": int(record.cycle_rank),
                    "polynomial": record.polynomial,
                    "cellCount": int(info["cell_count"]),
                    **geometry,
                }
                payload["regions"][region_key] = region_payload
                representative_rows.append({
                    "transition": spec["key"],
                    "title": spec["title"],
                    "mode": mode_key,
                    "mode_title": mode["title"],
                    "region_id": int(region_id),
                    "phase_id": phase_id,
                    "classic_region_ids": json.dumps(region_payload["classicRegionIds"]),
                    "classic_phase_ids": json.dumps(region_payload["classicPhaseIds"]),
                    "lambda": round(float(record.lam), 6),
                    "Gamma": round(float(record.gamma), 6),
                    "parameter_name": parameter_name,
                    "parameter_value": round(float(record.gamma), 6),
                    "cell_count": int(info["cell_count"]),
                    "source": record.source,
                    "core_mode": record.core_mode,
                    "nodes": int(record.nodes),
                    "edges": int(record.edges),
                    "components": int(record.components),
                    "cycle_rank": int(record.cycle_rank),
                    "polynomial": record.polynomial,
                    "surface_components": int(geometry["topology"]["components"]),
                    "surface_euler_characteristic": int(geometry["topology"]["euler_characteristic"]),
                    "surface_handle_rank": int(geometry["topology"]["handle_rank"]),
                    "surface_closed_in_window": bool(geometry["topology"]["closed_in_window"]),
                    "surface_touches_boundary": bool(geometry["topology"]["touches_boundary"]),
                    "surface_boundary_faces": json.dumps(geometry["topology"]["boundary_faces"]),
                    "surface_forces_vertex": bool(geometry["topology"]["forces_vertex"]),
                })
        payload["transitions"].append(transition_payload)

    bad_genus_zero_loops = [
        region["key"]
        for region in payload["regions"].values()
        if region["topology"]["closed_in_window"]
        and region["topology"]["handle_rank"] == 0
        and region["skeleton"]["edge_count"] > 0
    ]
    if bad_genus_zero_loops:
        raise AssertionError(f"Closed genus-zero surfaces displayed with non-vertex skeletons: {bad_genus_zero_loops[:5]}")

    payload["representatives"] = representative_rows
    QA_DATA_PATH.write_text(json.dumps(payload, separators=(",", ":")), encoding="utf-8")
    update_table_csv(QA_REPRESENTATIVES_PATH, "plotly_region_representatives", representative_rows)
    print("saved:", QA_DATA_PATH)
    print("saved:", QA_REPRESENTATIVES_PATH)
    return payload, representative_rows


def plotly_loader_html() -> str:
    try:
        import plotly.io as pio
        return "<script>" + pio.get_plotlyjs() + "</script>"
    except Exception:
        return '<script src="https://cdn.jsdelivr.net/npm/plotly.js-dist-min@2.35.2/plotly.min.js"></script>'


def yamada_display_text(polynomial: str) -> str:
    text = str(polynomial)
    chars = []
    for index, char in enumerate(text):
        if char != "A":
            chars.append(char)
            continue
        before = text[index - 1] if index else ""
        after = text[index + 1] if index + 1 < len(text) else ""
        if (not before.isalnum() and before != "_") and (not after.isalnum() and after != "_"):
            chars.append("Y")
        else:
            chars.append(char)
    return "".join(chars)


def payload_with_display_yamada(payload: dict) -> dict:
    html_payload = json.loads(json.dumps(payload))
    for region in html_payload.get("regions", {}).values():
        if "polynomial" in region:
            region["polynomial"] = yamada_display_text(region["polynomial"])
    for row in html_payload.get("representatives", []):
        if "polynomial" in row:
            row["polynomial"] = yamada_display_text(row["polynomial"])
    return html_payload


def write_interactive_plotly_html(payload: dict) -> Path:
    payload_json = json.dumps(payload_with_display_yamada(payload), separators=(",", ":"))
    html_template = """
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Hamiltonian Yamada geometry QA</title>
  __PLOTLY_LOADER__
  <style>
    :root { color-scheme: light; --ink:#111111; --muted:#555555; --line:#111111; --paper:#ffffff; --soft:#f4f4f4; }
    body { margin:0; background:var(--paper); color:var(--ink); font-family:"Times New Roman", Times, "DejaVu Serif", serif; }
    #app { max-width:1320px; margin:0 auto; padding:14px 16px 18px; }
    .topbar { display:flex; flex-wrap:wrap; align-items:center; gap:10px 18px; margin-bottom:10px; }
    .buttonGroup { display:flex; flex-wrap:wrap; align-items:center; gap:8px; }
    .buttonGroup::before { content:attr(data-label); font-size:12px; font-weight:700; color:var(--muted); text-transform:uppercase; letter-spacing:0.04em; margin-right:2px; font-family:Arial, Helvetica, sans-serif; }
    .topbar button { border:1px solid var(--line); background:var(--paper); color:var(--ink); padding:6px 10px; font-size:13px; cursor:pointer; font-family:Arial, Helvetica, sans-serif; }
    .topbar button[aria-pressed="true"] { background:var(--ink); color:var(--paper); }
    .layout { display:grid; grid-template-columns:minmax(380px, 0.92fr) minmax(440px, 1.08fr); gap:14px; align-items:start; }
    #phaseMap, #geometry { width:100%; height:640px; border:1.5px solid var(--line); box-sizing:border-box; }
    .status { min-height:78px; margin-top:10px; display:grid; grid-template-columns:1fr; gap:3px; font-size:13px; color:var(--muted); font-family:Arial, Helvetica, sans-serif; }
    .status strong { color:var(--ink); font-size:15px; }
    .poly { font-family:Menlo, Consolas, monospace; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; max-width:100%; }
    @media (max-width: 900px) { .layout { grid-template-columns:1fr; } #phaseMap, #geometry { height:520px; } }
  </style>
</head>
<body>
<div id="app">
  <div class="topbar">
    <div class="buttonGroup" id="modeButtons" data-label="Mode" aria-label="Phase modes"></div>
    <div class="buttonGroup" id="transitionButtons" data-label="Transition" aria-label="Transitions"></div>
  </div>
  <div class="layout">
    <div>
      <div id="phaseMap" aria-label="Clickable phase map"></div>
    </div>
    <div>
      <div id="geometry" aria-label="Representative exceptional surface and skeleton"></div>
      <div class="status" id="status" aria-live="polite"></div>
    </div>
  </div>
</div>
<script>
const payload = __PAYLOAD__;
const transitions = payload.transitions;
const regions = payload.regions;
const modes = payload.modes;
let activeTransition = transitions[0].key;
let activeMode = modes[0].key;

function discreteColorscale(colors) {
  const scale = [];
  const n = colors.length;
  if (n === 1) return [[0, colors[0]], [1, colors[0]]];
  colors.forEach((color, index) => {
    scale.push([index / n, color]);
    scale.push([(index + 1) / n, color]);
  });
  return scale;
}

function transitionByKey(key) {
  return transitions.find(item => item.key === key) || transitions[0];
}

function modeByKey(key) {
  return modes.find(item => item.key === key) || modes[0];
}

function modeStatsLine(modeKey, data) {
  if (modeKey === "classic") {
    return `${data.classes} Classic Yamada classes, ${data.components} connected regions; raw ${data.rawClasses}`;
  }
  return `${data.classes} contraction classes, ${data.components} connected regions; merged ${data.contractionMergedClassicRegions} Classic regions`;
}

function htmlTransitionTitle(item, mode) {
  const startName = item.startName || "0";
  const endName = item.endName || "1";
  const suffix = mode.key === "classic" ? "" : ` (${mode.short_title || mode.title})`;
  return `(1−λ)<i>H</i><sub>${startName}</sub>+λ<i>H</i><sub>${endName}</sub>${suffix}`;
}

function edgeCoordinates(values) {
  if (values.length === 1) return [values[0] - 0.5, values[0] + 0.5];
  const edges = [];
  edges.push(values[0] - 0.5 * (values[1] - values[0]));
  for (let index = 1; index < values.length; index += 1) {
    edges.push(0.5 * (values[index - 1] + values[index]));
  }
  edges.push(values[values.length - 1] + 0.5 * (values[values.length - 1] - values[values.length - 2]));
  return edges;
}

function phaseBoundaryTrace(xValues, yValues, labels) {
  const xEdges = edgeCoordinates(xValues);
  const yEdges = edgeCoordinates(yValues);
  const xs = [];
  const ys = [];
  for (let row = 0; row < labels.length; row += 1) {
    for (let col = 0; col < labels[row].length - 1; col += 1) {
      if (labels[row][col] === labels[row][col + 1]) continue;
      xs.push(xEdges[col + 1], xEdges[col + 1], null);
      ys.push(yEdges[row], yEdges[row + 1], null);
    }
  }
  for (let row = 0; row < labels.length - 1; row += 1) {
    for (let col = 0; col < labels[row].length; col += 1) {
      if (labels[row][col] === labels[row + 1][col]) continue;
      xs.push(xEdges[col], xEdges[col + 1], null);
      ys.push(yEdges[row + 1], yEdges[row + 1], null);
    }
  }
  return {
    type: "scatter",
    mode: "lines",
    x: xs,
    y: ys,
    line: { color: "#111111", width: 1.2 },
    hoverinfo: "skip",
    showlegend: false,
    name: "phase boundary"
  };
}

function plotPhaseMap(transitionKey = activeTransition, modeKey = activeMode) {
  const item = transitionByKey(transitionKey);
  const mode = modeByKey(modeKey);
  const data = item.modes[mode.key];
  activeTransition = item.key;
  activeMode = mode.key;
  const trace = {
    type: "heatmap",
    x: item.lambdas || payload.lambdas,
    y: item.parameters || item.gammas,
    z: data.z,
    customdata: data.regionKeys,
    zmin: 0.5,
    zmax: data.colors.length + 0.5,
    colorscale: discreteColorscale(data.colors),
    showscale: true,
    colorbar: {
      title: { text: "Υ", side: "top", font: { family: "Times New Roman, Times, serif", size: 22, color: "#111111" } },
      tickmode: "array",
      tickvals: [],
      ticks: "",
      showticklabels: false,
      outlinecolor: "#111111",
      outlinewidth: 1.2,
      thickness: 18,
      len: 0.86
    },
    hovertemplate: `λ=%{x:.3f}<br>E=%{y:.3f}<br>region=%{customdata}<br>phase=%{z}<extra></extra>`
  };
  const xValues = item.lambdas || payload.lambdas;
  const yValues = item.parameters || item.gammas;
  const yLabel = "E";
  const boundaryTrace = phaseBoundaryTrace(xValues, yValues, data.z);
  const layout = {
    title: {
      text: htmlTransitionTitle(item, mode),
      x: 0.5,
      xanchor: "center",
      font: { family: "Times New Roman, Times, serif", size: 24, color: "#111111" }
    },
    margin: { l: 74, r: 72, t: 68, b: 72 },
    paper_bgcolor: "#ffffff",
    plot_bgcolor: "#ffffff",
    font: { family: "Times New Roman, Times, serif", color: "#111111" },
    xaxis: { title: { text: "λ", font: { size: 28, family: "Times New Roman, Times, serif" } }, range: [xValues[0], xValues[xValues.length - 1]], mirror: true, ticks: "outside", tickfont: { size: 18 }, linewidth: 2, linecolor: "#111111", showgrid: false, zeroline: false },
    yaxis: { title: { text: yLabel, font: { size: 28, family: "Times New Roman, Times, serif" } }, range: [yValues[0], yValues[yValues.length - 1]], mirror: true, ticks: "outside", tickfont: { size: 18 }, linewidth: 2, linecolor: "#111111", showgrid: false, zeroline: false }
  };
  Plotly.react("phaseMap", [trace, boundaryTrace], layout, { responsive: true, displaylogo: false });
  const phaseMap = document.getElementById("phaseMap");
  phaseMap.removeAllListeners?.("plotly_click");
  phaseMap.on("plotly_click", event => {
    const regionKey = event.points?.[0]?.customdata;
    if (regionKey) selectRegion(regionKey);
  });
  const firstRegion = Object.keys(regions).find(regionKey => regionKey.startsWith(`${mode.key}:${item.key}:`));
  if (firstRegion) selectRegion(firstRegion);
}

function geometryTraces(region) {
  const traces = [];
  if (region.surface && region.surface.triangle_count) {
    traces.push({
      type: "mesh3d",
      name: "exceptional surface",
      x: region.surface.x,
      y: region.surface.y,
      z: region.surface.z,
      i: region.surface.i,
      j: region.surface.j,
      k: region.surface.k,
      color: region.color,
      opacity: 0.42,
      flatshading: false,
      hoverinfo: "skip"
    });
  }
  if (region.skeleton.lines.x.length) {
    traces.push({
      type: "scatter3d",
      name: "skeleton",
      x: region.skeleton.lines.x,
      y: region.skeleton.lines.y,
      z: region.skeleton.lines.z,
      mode: "lines",
      line: { color: "#111111", width: 8 },
      hoverinfo: "skip"
    });
  }
  traces.push({
    type: "scatter3d",
    name: "vertices",
    x: region.skeleton.nodes.x,
    y: region.skeleton.nodes.y,
    z: region.skeleton.nodes.z,
    mode: "markers",
    marker: { size: 4, color: "#d62728", line: { color: "#111111", width: 1 } },
    hovertemplate: "core vertex<extra></extra>"
  });
  return traces;
}

function compactIdList(values, limit = 8) {
  if (!values || values.length === 0) return "none";
  if (values.length <= limit) return values.join(", ");
  return `${values.slice(0, limit).join(", ")} +${values.length - limit} more`;
}

function topologyStatusLine(topology) {
  if (!topology) return "Surface topology: unavailable";
  const faces = topology.boundary_faces && topology.boundary_faces.length ? topology.boundary_faces.join(", ") : "none";
  if (topology.touches_boundary) {
    return `Surface topology: boundary-open in this k-window; voxel handle rank=${topology.handle_rank}, components=${topology.components}, Euler=${topology.euler_characteristic}; touches ${faces}`;
  }
  return `Surface topology: closed in this k-window; voxel handle rank=${topology.handle_rank}, components=${topology.components}, Euler=${topology.euler_characteristic}`;
}

function yamadaDisplay(polynomial) {
  return String(polynomial || "").replace(/(^|[^A-Za-z0-9_])A(?![A-Za-z0-9_])/g, "$1Y");
}

function selectRegion(regionKey) {
  const region = regions[regionKey];
  if (!region) return;
  const layout = {
    title: {
      text: `${region.title}: ${region.modeTitle}, region ${region.regionId}, phase ${region.phaseId}`,
      x: 0,
      xanchor: "left",
      font: { family: "Times New Roman, serif", size: 21, color: "#111111" }
    },
    margin: { l: 0, r: 0, t: 50, b: 0 },
    paper_bgcolor: "#ffffff",
    plot_bgcolor: "#ffffff",
    showlegend: true,
    legend: { x: 0.02, y: 0.98, bgcolor: "rgba(255,255,255,0.75)" },
    scene: {
      xaxis: { title: "kₓ", showbackground: false, gridcolor: "#dddddd", zerolinecolor: "#cccccc" },
      yaxis: { title: "kᵧ", showbackground: false, gridcolor: "#dddddd", zerolinecolor: "#cccccc" },
      zaxis: { title: "k_z", showbackground: false, gridcolor: "#dddddd", zerolinecolor: "#cccccc" },
      aspectmode: "data",
      camera: { eye: { x: 1.45, y: 1.55, z: 1.15 } }
    }
  };
  const skeletonKind = region.skeleton.edge_count ? "edge skeleton" : "vertex-only skeleton";
  const parameterLabel = "E";
  const parameterValue = region.parameterValue ?? region.Gamma;
  Plotly.react("geometry", geometryTraces(region), layout, { responsive: true, displaylogo: false });
  document.getElementById("status").innerHTML = `
    <strong>${region.title} - ${region.modeTitle}</strong>
    <span>region ${region.regionId}, phase ${region.phaseId}; representative λ=${region.lambda}, ${parameterLabel}=${parameterValue}; ${region.source}, ${region.coreMode}; ${skeletonKind}; core V=${region.nodes}, E=${region.edges}, β=${region.cycleRank}; surface triangles=${region.surface.triangle_count}</span>
    <span>${region.startName ? `endpoints: ${region.startName} → ${region.endName}` : ""}</span>
    <span>${topologyStatusLine(region.topology)}</span>
    <span>Classic regions: ${compactIdList(region.classicRegionIds)}; Classic phases: ${compactIdList(region.classicPhaseIds)}; cells=${region.cellCount}</span>
    <span class="poly">Yamada: ${yamadaDisplay(region.polynomial)}</span>
  `;
}

function syncButtons() {
  document.querySelectorAll("#modeButtons button").forEach(button => {
    button.setAttribute("aria-pressed", button.dataset.key === activeMode ? "true" : "false");
  });
  document.querySelectorAll("#transitionButtons button").forEach(button => {
    button.setAttribute("aria-pressed", button.dataset.key === activeTransition ? "true" : "false");
  });
}

function buildButtons() {
  const modeHolder = document.getElementById("modeButtons");
  modeHolder.innerHTML = "";
  modes.forEach(mode => {
    const button = document.createElement("button");
    button.type = "button";
    button.dataset.key = mode.key;
    button.textContent = mode.title;
    button.setAttribute("aria-pressed", mode.key === activeMode ? "true" : "false");
    button.addEventListener("click", () => {
      activeMode = mode.key;
      syncButtons();
      plotPhaseMap(activeTransition, activeMode);
    });
    modeHolder.appendChild(button);
  });

  const transitionHolder = document.getElementById("transitionButtons");
  transitionHolder.innerHTML = "";
  transitions.forEach(item => {
    const button = document.createElement("button");
    button.type = "button";
    button.dataset.key = item.key;
    button.textContent = item.title;
    button.setAttribute("aria-pressed", item.key === activeTransition ? "true" : "false");
    button.addEventListener("click", () => {
      activeTransition = item.key;
      syncButtons();
      plotPhaseMap(activeTransition, activeMode);
    });
    transitionHolder.appendChild(button);
  });
}

buildButtons();
plotPhaseMap(activeTransition, activeMode);
window.addEventListener("resize", () => {
  Plotly.Plots.resize(document.getElementById("phaseMap"));
  Plotly.Plots.resize(document.getElementById("geometry"));
});
</script>
</body>
</html>
"""
    html_text = (
        html_template
        .replace("__PLOTLY_LOADER__", plotly_loader_html())
        .replace("__PAYLOAD__", payload_json)
    )
    QA_HTML_PATH.write_text(html_text, encoding="utf-8")
    print("saved:", QA_HTML_PATH)
    return QA_HTML_PATH


interactive_payload, interactive_representatives = build_interactive_payload()
interactive_html_path = write_interactive_plotly_html(interactive_payload)
print("interactive representative regions:", len(interactive_payload["regions"]))
display(IFrame(src=interactive_html_path.as_uri(), width="100%", height=840))


## Final phase-map verification

The final cell checks package provenance, transition coverage, error totals, and
the existence of the generated interactive artifact. A passing final cell means
the configured finite-grid workflow completed its stated checks; it does not
replace resolution convergence or an analytic phase-boundary proof.


In [ ]:
print("knotted_graph source:", Path(knotted_graph.__file__).resolve())
print("repository root:", ROOT)
print("transitions:", [spec["key"] for spec in TRANSITIONS])
print("total displayed cells:", len(all_records))
print("phase-map errors:", sum(record.error is not None for record in all_records))
print("interactive HTML:", QA_HTML_PATH)

assert knotted_graph.__version__.startswith("0.2"), (
    "This reproduction notebook requires the current 0.2 development API."
)
assert all(spec["key"] in all_records_by_transition for spec in TRANSITIONS)
assert QA_HTML_PATH.exists(), (
    "Run the interactive Plotly cell above to regenerate the final HTML artifact."
)

print("FINAL NOTEBOOK CHECK: PASS")
